# Ανάλυση Ιατρικών Εικόνων DICOM στην Python


---

**Δομή του μαθήματος:**

| Μέρος | Ενότητες | Περιεχόμενο |
|-------|----------|--------------|
| **A** | 1–25 | Βασικές έννοιες DICOM, ανάγνωση, tags, οπτικοποίηση, Hounsfield Units, windowing, σειρές, εργαλεία |
| **B** | 26–36 | Στατιστική ανάλυση, ιστογράμματα, ποιότητα εικόνας, θόρυβος, αναφορές |

**Τι θα μάθετε σε αυτό το notebook:**
- Τι είναι ένα αρχείο DICOM και γιατί υπάρχει.
- Πώς διαβάζουμε DICOM με την Python και τη βιβλιοθήκη `pydicom`.
- Πώς εξερευνούμε metadata (DICOM tags) — δημογραφικά ασθενούς, παράμετροι εξέτασης, χωρικές πληροφορίες.
- Πώς οπτικοποιούμε ιατρικές εικόνες σωστά (grayscale, windowing, Hounsfield).
- Πώς πλοηγούμαστε σε σειρές εικόνων και χτίζουμε 3D όγκους.
- Πώς αναλύουμε στατιστικά την εικόνα και αξιολογούμε ποιότητα.

**Τι ΧΡΕΙΑΖΕΤΑΙ να ξέρετε από πριν:**
- Βασικά Python (μεταβλητές, συναρτήσεις, κλάσεις).
- Στοιχειώδης χρήση Jupyter notebooks.

> 💡 **Συμβουλή του διδάσκοντα:** Μην απλώς τρέχετε τα κελιά. Σταματήστε σε καθένα, διαβάστε την ελληνική επεξήγηση, και προσπαθήστε να αναγνωρίσετε τι κάνει ο κώδικας **πριν** δείτε το αποτέλεσμα. Έτσι θα μάθετε πραγματικά.


In [ ]:
"""
This notebook introduces students to DICOM (Digital Imaging and Communications in Medicine) 
images - the standard format for medical imaging data.

Learning Objectives:
1. Understand what DICOM files are
2. Load and read DICOM images
3. Explore DICOM tags and metadata
4. Visualize medical images
5. Navigate through image series/slices
6. Perform basic image processing
"""


## Ενότητα 1 — Εγκατάσταση Πακέτων & Εισαγωγή Βιβλιοθηκών

### Γιατί χρειαζόμαστε εξωτερικές βιβλιοθήκες;

Η Python μόνη της δεν ξέρει τι είναι DICOM, ούτε πώς να φτιάξει διαγράμματα, ούτε πώς να κάνει αποδοτικές πράξεις σε εκατομμύρια αριθμούς. Αυτές τις δουλειές τις αναλαμβάνουν εξειδικευμένες **βιβλιοθήκες** που τις «εισάγουμε» (import) στο πρόγραμμά μας.

### Τι κάνει η κάθε βιβλιοθήκη

| Βιβλιοθήκη | Ρόλος | Τυπική χρήση |
|------------|-------|--------------|
| **`pydicom`** | Ανάγνωση/εγγραφή DICOM | `pydicom.dcmread()` |
| **`numpy`** (ως `np`) | Αριθμητικοί πίνακες (arrays) | Πράξεις σε pixel |
| **`matplotlib.pyplot`** (ως `plt`) | Γραφικά | `plt.imshow()`, `plt.hist()` |
| **`pandas`** (ως `pd`) | Πίνακες δεδομένων (DataFrames) | Λίστες tags, αναφορές |
| **`pathlib`** / **`os`** | Διαχείριση αρχείων και διαδρομών | Φόρτωση φακέλων |
| **`IPython.display`** | Ωραίες εμφανίσεις στο Jupyter | `display(df)` για πίνακες |

### Πώς εγκαθίστανται;

Αν δεν έχετε ήδη τα πακέτα, ξεκομμένη το σχόλιο στη γραμμή `# !pip install ...` και τρέξτε το κελί. Το `!` λέει στο Jupyter να εκτελέσει εντολή **shell**, όχι Python.

### Τι κάνουν οι ρυθμίσεις `plt.rcParams`

Είναι **καθολικές προεπιλογές** για όλα τα διαγράμματα του notebook:

- `figure.figsize = (12, 8)` — μεγάλα γραφήματα από προεπιλογή.
- `image.cmap = 'gray'` — όλες οι εικόνες σε **κλίμακα του γκρι**, που είναι το πρότυπο στις ιατρικές εικόνες.

> 📌 **Σημείωση:** Στο ιατρικό περιβάλλον, μην χρησιμοποιείτε χρωματικές κλίμακες (rainbow, jet) για τις ίδιες τις εικόνες — δημιουργούν ψευδαισθήσεις δομών που δεν υπάρχουν. Η γκρι κλίμακα είναι ο χρυσός κανόνας.


In [ ]:
# First, let's install required packages (run this cell first!)
# Uncomment the line below if packages are not installed
# !pip install pydicom numpy matplotlib pillow pandas

import pydicom
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML
import os
import json

# Set up matplotlib for better visualization
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.cmap'] = 'gray'

print("✓ All packages imported successfully!")
print(f"PyDICOM version: {pydicom.__version__}")


## Ενότητα 2 — Τι είναι το DICOM;

### Από το χάος στο πρότυπο

Πριν το DICOM, κάθε κατασκευαστής ιατρικού μηχανήματος (Siemens, GE, Philips...) είχε **δικό του format** για εικόνες. Νοσοκομεία με μηχανήματα από διαφορετικές εταιρείες δεν μπορούσαν να ανταλλάξουν δεδομένα. Το **DICOM** (Digital Imaging and Communications in Medicine) δημιουργήθηκε από τους ACR και NEMA τη δεκαετία του '80 ακριβώς για να λύσει αυτό το πρόβλημα. Σήμερα είναι το **παγκόσμιο πρότυπο**.

### Γιατί το DICOM δεν είναι «απλά μια εικόνα»;

Ένα αρχείο JPG είναι μόνο pixels. Ένα αρχείο DICOM περιέχει **τρεις συνιστώσες**:

1. **Δεδομένα εικόνας** — οι τιμές των pixel (ή voxel σε 3D).
2. **Metadata (DICOM tags)** — εκατοντάδες πεδία που περιγράφουν τα πάντα: ποιος ο ασθενής, ποιο μηχάνημα, ποια ημερομηνία, ποιες παράμετροι, ποια θέση στο σώμα...
3. **Header** — τεχνικές πληροφορίες για την κωδικοποίηση του αρχείου.

> 🎯 **Κρίσιμη σκέψη:** Σε ένα πραγματικό κλινικό σύστημα, αν χάσετε τα metadata δεν μπορείτε να εμπιστευθείτε την εικόνα. Δεν ξέρετε σε ποιον ανήκει, πώς τραβήχτηκε, σε ποια θέση. Χωρίς context, μια εικόνα είναι ιατρικά **άχρηστη**.

### Τα modalities που υποστηρίζει

Όλα τα ιατρικά απεικονιστικά μηχανήματα παράγουν DICOM:

- **CT** (Υπολογιστική Τομογραφία)
- **MRI / MR** (Μαγνητική Τομογραφία)
- **CR / DX / RF** (Ψηφιακή Ακτινογραφία / Φθοριοσκόπηση)
- **US** (Υπερηχογράφημα)
- **PET / NM** (Τομογραφία Εκπομπής Ποζιτρονίων / Πυρηνική Ιατρική)
- **MG** (Μαστογραφία)
- **OCT, Endoscopy, RT...**

### Παρατηρήστε στον κώδικα

Το κελί απλώς **τυπώνει** μια περιγραφή. Δεν εκτελεί τίποτα τεχνικά. Είναι κελί «θεωρίας» — να το διαβάσετε, όχι να ψάχνετε αλγόριθμο μέσα του.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                    WHAT IS DICOM?                              ║
╚════════════════════════════════════════════════════════════════╝

DICOM (Digital Imaging and Communications in Medicine) is:
- The international standard for medical images and related information
- More than just an image format - it contains metadata about the patient,
  study, and acquisition parameters
- Used by virtually all medical imaging modalities:
  * CT (Computed Tomography)
  * MRI (Magnetic Resonance Imaging)
  * X-Ray
  * Ultrasound
  * PET (Positron Emission Tomography)
  * And many more...

Key Components:
1. Image Data: The actual pixel/voxel values
2. DICOM Tags: Metadata describing the image
3. Header: Information about patient, study, series, equipment
""")


## Ενότητα 3 — Από πού παίρνουμε δείγματα DICOM;

### Τρεις πηγές δεδομένων

Για να εξασκηθείτε χρειάζεστε αρχεία DICOM. Υπάρχουν τρεις κύριες πηγές:

#### 1. Test data της `pydicom`
Η βιβλιοθήκη φορτώνεται μαζί με μερικά **μικρά test αρχεία** (CT, MRI, RT plan κ.ά.) — ιδανικά για διδακτικούς σκοπούς. Τα παίρνουμε με:

```python
from pydicom.data import get_testdata_file
path = get_testdata_file("CT_small.dcm")
```

#### 2. Δημόσια ερευνητικά datasets
- **TCIA (The Cancer Imaging Archive)** — εκατοντάδες χιλιάδες ανωνυμοποιημένες σαρώσεις από κλινικές μελέτες. Ιδανικό για ML projects.
- **Medical Segmentation Decathlon** — datasets με segmentation labels.
- **OpenNeuro, ADNI, OASIS** — εξειδικευμένα για νευροαπεικόνιση.
- **Grand Challenge** — datasets από διαγωνισμούς ML.

#### 3. Τα δικά σας DICOM
Αν έχετε **πραγματικά κλινικά δεδομένα** από νοσοκομείο, χρειάζεστε:
- Έγκριση από **Επιτροπή Δεοντολογίας / IRB**.
- **Ανωνυμοποίηση** (την κάνουμε στην Ενότητα 21).
- Συμμόρφωση με **GDPR** (στην Ευρώπη) και τοπικούς κανονισμούς.

> ⚠️ **ΣΗΜΑΝΤΙΚΟ — Ποτέ μην:**
> - Ανεβάζετε κλινικά DICOM σε δημόσια GitHub repositories.
> - Στέλνετε δεδομένα ασθενών σε cloud υπηρεσίες χωρίς συμβατικό πλαίσιο.
> - Δημοσιεύετε εικόνες χωρίς να αφαιρέσετε **ΟΛΑ** τα PHI (Protected Health Information).

### Τι κάνει ο κώδικας

Φορτώνει το «CT_small.dcm», ένα μικρό demo αρχείο που έρχεται με την pydicom. Η μεταβλητή `sample_dcm_path` είναι απλά η **διαδρομή** στο αρχείο — δεν έχουμε ακόμη ανοίξει το περιεχόμενό του. Αυτό γίνεται στην επόμενη ενότητα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              GETTING SAMPLE DICOM FILES                        ║
╚════════════════════════════════════════════════════════════════╝

For this tutorial, you can:
1. Use pydicom's built-in sample files
2. Download from public datasets (e.g., The Cancer Imaging Archive)
3. Use your own DICOM files

We'll use pydicom's built-in samples for demonstration.
""")

# Load a sample DICOM file from pydicom's test data
from pydicom.data import get_testdata_file

# Get sample DICOM file
sample_dcm_path = get_testdata_file("CT_small.dcm")
print(f"Sample DICOM file loaded from: {sample_dcm_path}")


## Ενότητα 4 — Ανάγνωση Αρχείων DICOM

### Η συνάρτηση «κλειδί»: `pydicom.dcmread`

Όλη η μαγεία ξεκινά με μία γραμμή:

```python
dicom_data = pydicom.dcmread(sample_dcm_path)
```

Η `dcmread` διαβάζει το αρχείο DICOM και επιστρέφει ένα **αντικείμενο τύπου `FileDataset`**. Αυτό το αντικείμενο συμπεριφέρεται με δύο τρόπους ταυτόχρονα:

- **Σαν «αντικείμενο» με ιδιότητες:** `dicom_data.PatientName`
- **Σαν «λεξικό» με κλειδιά:** `dicom_data[0x0008, 0x0060]`

Θα δούμε και τους δύο τρόπους στις επόμενες ενότητες.

### Lazy loading — γιατί είναι γρήγορο

Παρατηρήστε ότι τα **pixel** δεν φορτώνονται αμέσως στη μνήμη. Η `dcmread` φορτώνει μόνο τα **metadata** και «δείχνει» πού είναι τα pixel στο αρχείο. Τα pixel διαβάζονται **όταν τα ζητήσετε** (μέσω `dicom_data.pixel_array`). Αυτό λέγεται **lazy evaluation** και είναι πολύ χρήσιμο όταν έχετε χιλιάδες αρχεία και θέλετε να φιλτράρετε με βάση τα tags χωρίς να φορτώνετε τα pixel.

> 💡 **Tip για ταχύτητα:** Αν θέλετε να διαβάσετε **μόνο** τα metadata (όχι τα pixel), χρησιμοποιήστε:
> ```python
> dcm = pydicom.dcmread(path, stop_before_pixels=True)
> ```
> Αυτό είναι 10× πιο γρήγορο για μεγάλα αρχεία.

### Τι θα δείτε στην έξοδο

- Επιβεβαίωση ότι το αρχείο φορτώθηκε (✓).
- Ο τύπος του αντικειμένου: `<class 'pydicom.dataset.FileDataset'>`.

> 🧠 **Συνηθισμένα σφάλματα ανάγνωσης:**
> - **`InvalidDicomError`** — το αρχείο δεν είναι έγκυρο DICOM. Δοκιμάστε `force=True` αν λείπει το preamble.
> - **`PermissionError`** — δεν έχετε δικαιώματα ανάγνωσης. Ελέγξτε τα permissions του αρχείου.
> - **`FileNotFoundError`** — λάθος διαδρομή.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  READING DICOM FILES                           ║
╚════════════════════════════════════════════════════════════════╝
""")

# Read the DICOM file
dicom_data = pydicom.dcmread(sample_dcm_path)

print("✓ DICOM file successfully loaded!")
print(f"Type of object: {type(dicom_data)}")


## Ενότητα 5 — Κατανόηση των DICOM Tags

### Τι είναι ένα tag;

Ένα **DICOM tag** είναι ένα στοιχείο metadata. Σκεφτείτε ένα τεράστιο τυποποιημένο ερωτηματολόγιο που γεμίζει αυτόματα το μηχάνημα κάθε φορά που τραβάει εικόνα. Κάθε ερώτηση («όνομα ασθενούς», «kV εκπομπής», «πάχος τομής»...) είναι ένα tag.

### Δομή ενός tag

Κάθε tag έχει **τρία συστατικά**:

| Συστατικό | Παράδειγμα | Σημασία |
|-----------|------------|---------|
| **Tag number** | `(0008, 0060)` | Δύο 16-bit ακέραιοι σε hex: (group, element) |
| **VR (Value Representation)** | `CS`, `PN`, `DA`, `IS`... | Ο τύπος δεδομένων |
| **Value** | `"CT"` | Η ίδια η τιμή |

### Οι τύποι (VRs) που θα συναντήσετε

| VR | Σημασία | Παράδειγμα |
|----|---------|------------|
| **PN** | Person Name | `Doe^John` |
| **CS** | Code String | `CT`, `MR`, `MONOCHROME2` |
| **DA** | Date | `20240315` |
| **TM** | Time | `143025.000` |
| **IS** | Integer String | `"512"` |
| **DS** | Decimal String | `"0.7"` |
| **UI** | Unique Identifier | `1.2.840.10008...` |
| **OB / OW** | Other Byte/Word (binary) | Pixel data, ιδιωτικά |

### Πώς οργανώνονται τα tags — τα modules

Το πρότυπο DICOM οργανώνει τα tags σε **modules**:

- **Patient Module** — όνομα, ID, ηλικία, φύλο.
- **Study Module** — ημερομηνία/περιγραφή/UID της εξέτασης.
- **Series Module** — ομάδα τομών μιας ακολουθίας.
- **Image Module** — διαστάσεις, encoding, position.
- **Equipment Module** — κατασκευαστής, μοντέλο, software.
- **Modality-specific** (CT Image, MR Image, ...) — παράμετροι για το συγκεκριμένο τύπο.

### Τι κάνει ο κώδικας

Με `print(dicom_data)` τυπώνεται **όλη η DICOM header** — δηλαδή κάθε tag του αρχείου, με όλα τα συστατικά του. Είναι μεγάλη έξοδος. **Ξοδέψτε χρόνο** να την κοιτάξετε — είναι ο καλύτερος τρόπος να καταλάβετε τι περιέχει ένα DICOM.

> 📚 **Reference:** Όλα τα tags που έχει ορίσει το πρότυπο βρίσκονται στο επίσημο **DICOM Data Dictionary** στη διεύθυνση `https://dicom.innolitics.com/ciods` — βάλτε το στα bookmarks σας.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  UNDERSTANDING DICOM TAGS                      ║
╚════════════════════════════════════════════════════════════════╝

DICOM tags are metadata elements that describe the image.
Each tag has:
- Tag number: (XXXX, XXXX) in hexadecimal
- VR (Value Representation): data type
- Value: the actual data
""")

# Display the entire DICOM header
print("\n" + "="*70)
print("COMPLETE DICOM HEADER:")
print("="*70)
print(dicom_data)


## Ενότητα 6 — Πρόσβαση σε Συγκεκριμένα Tags

### Τα τρία βασικά «modules» που μας ενδιαφέρουν συνήθως

Σε αυτή την ενότητα διαβάζουμε επιλεκτικά **τρεις ομάδες tags** που χρησιμοποιούνται σχεδόν σε κάθε ανάλυση:

#### 1. Patient Module — δημογραφικά
- `PatientName` — όνομα (μορφή `LastName^FirstName^Middle`).
- `PatientID` — μοναδικό ID στο νοσοκομείο.
- `PatientBirthDate` — ημερομηνία γέννησης.
- `PatientSex` — `M`, `F`, `O`.

#### 2. Study Module — η εξέταση
- `StudyDate` — πότε έγινε.
- `StudyDescription` — περιγραφή («Brain MRI without contrast»).
- `StudyInstanceUID` — μοναδικό αναγνωριστικό. Δύο εικόνες με το ίδιο UID ανήκουν στην **ίδια εξέταση**.

#### 3. Image Module — η εικόνα
- `Modality` — `CT`, `MR`, `US`, ...
- `Rows`, `Columns` — διαστάσεις.
- `SliceThickness` — πάχος τομής σε mm.
- `PixelSpacing` — απόσταση μεταξύ pixel σε mm.

### Γιατί χρειαζόμαστε `try / except`;

Παρατηρήστε ότι ο κώδικας τυλίγει την προσπάθεια ανάγνωσης σε `try`/`except AttributeError`. Γιατί;

> ⚠️ **Όχι όλα τα DICOM έχουν όλα τα tags!** Ορισμένα tags είναι «type 1» (υποχρεωτικά πάντα), «type 2» (υποχρεωτικά αλλά μπορεί να είναι κενά), και «type 3» (προαιρετικά). Επιπλέον, σε ανωνυμοποιημένα δεδομένα, τα δημογραφικά μπορεί να έχουν αφαιρεθεί.

Αν προσπαθήσετε `dicom_data.PatientAge` και το tag δεν υπάρχει, η Python ρίχνει `AttributeError`. Με το `try/except` το πρόγραμμά μας **δεν κρασάρει** — απλώς εμφανίζει «Some patient information not available».

### Η εναλλακτική: `hasattr()`

Στις τελευταίες γραμμές βλέπετε:
```python
if hasattr(dicom_data, 'SliceThickness'):
    print(f"Slice Thickness: {dicom_data.SliceThickness} mm")
```

Αυτό είναι **πιο καθαρό** από `try/except` όταν θέλετε απλώς να ελέγξετε αν υπάρχει το tag πριν το χρησιμοποιήσετε. Στην επόμενη ενότητα θα δούμε και την `get()` που είναι ακόμα πιο κομψή.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              ACCESSING SPECIFIC DICOM TAGS                     ║
╚════════════════════════════════════════════════════════════════╝
""")

# Common DICOM tags and how to access them
print("\n📋 PATIENT INFORMATION:")
print("-" * 50)
try:
    print(f"Patient Name: {dicom_data.PatientName}")
    print(f"Patient ID: {dicom_data.PatientID}")
    print(f"Patient Birth Date: {dicom_data.PatientBirthDate}")
    print(f"Patient Sex: {dicom_data.PatientSex}")
except AttributeError as e:
    print(f"Some patient information not available: {e}")

print("\n🏥 STUDY INFORMATION:")
print("-" * 50)
try:
    print(f"Study Date: {dicom_data.StudyDate}")
    print(f"Study Description: {dicom_data.StudyDescription}")
    print(f"Study Instance UID: {dicom_data.StudyInstanceUID}")
except AttributeError as e:
    print(f"Some study information not available: {e}")

print("\n📸 IMAGE ACQUISITION PARAMETERS:")
print("-" * 50)
print(f"Modality: {dicom_data.Modality}")
print(f"Rows (Height): {dicom_data.Rows}")
print(f"Columns (Width): {dicom_data.Columns}")

# Check if these attributes exist before accessing
if hasattr(dicom_data, 'SliceThickness'):
    print(f"Slice Thickness: {dicom_data.SliceThickness} mm")
if hasattr(dicom_data, 'PixelSpacing'):
    print(f"Pixel Spacing: {dicom_data.PixelSpacing} mm")


## Ενότητα 7 — Τέσσερις Τρόποι Πρόσβασης σε ένα Tag

### Το ίδιο αποτέλεσμα, διαφορετικός κώδικας

Στη Python (όπως και στη ζωή), συχνά υπάρχουν πολλοί τρόποι να πετύχετε το ίδιο πράγμα. Σε αυτή την ενότητα βλέπετε **τέσσερις τρόπους** να πάρετε την τιμή του tag `Modality`:

| # | Μέθοδος | Πότε προτιμάται |
|---|---------|------------------|
| 1 | `dicom_data.Modality` | Γρήγορη, καθαρή — όταν είστε σίγουροι ότι το tag υπάρχει |
| 2 | `dicom_data[0x0008, 0x0060].value` | Όταν δεν θυμάστε το όνομα αλλά ξέρετε τον αριθμό |
| 3 | `dicom_data.get("Modality", "Not found")` | **Συνιστάται** — δεν κρασάρει αν λείπει το tag |
| 4 | `dicom_data.data_element("Modality").value` | Όταν θέλετε ολόκληρο το element (όχι μόνο την τιμή) |

### Σύγκριση

#### Μέθοδος 1: Direct attribute
```python
modality = dicom_data.Modality
```
**Πλεονεκτήματα:** Σύντομο, ευανάγνωστο.
**Μειονεκτήματα:** Κρασάρει με `AttributeError` αν το tag δεν υπάρχει.

#### Μέθοδος 2: Tag number (hex)
```python
modality = dicom_data[0x0008, 0x0060].value
```
**Πλεονεκτήματα:** Λειτουργεί ακόμα και για **ιδιωτικά tags** (private tags) που δεν έχουν κανονικό όνομα.
**Μειονεκτήματα:** Δυσανάγνωστο. Πρέπει να ξέρετε τον αριθμό.

#### Μέθοδος 3: `get()` με default
```python
modality = dicom_data.get("Modality", "Not found")
```
**Πλεονεκτήματα:** **Ποτέ** δεν κρασάρει. Επιστρέφει το default αν λείπει.
**Μειονεκτήματα:** Λίγο πιο φλύαρο.

#### Μέθοδος 4: `data_element()`
```python
elem = dicom_data.data_element("Modality")
modality = elem.value
print(elem.VR)        # επίσης ο τύπος
print(elem.tag)       # επίσης ο αριθμός
```
**Πλεονεκτήματα:** Επιστρέφει ολόκληρο το element με όλα τα metadata του (VR, tag number).
**Μειονεκτήματα:** Δύο γραμμές αντί για μία.

### Συμβουλή του διδάσκοντα

> 🎯 **Στη ρουτίνα σας, χρησιμοποιείτε τη Μέθοδο 3 (`.get()`) για παραγωγικό κώδικα.** Είναι ο μόνος τρόπος που εγγυάται ότι το πρόγραμμά σας δεν θα σταματήσει σε κάποιο «παράξενο» αρχείο μέσα σε 1000+ DICOMs.

> 💡 **Παράδειγμα defensive programming:**
> ```python
> patient_age = dicom_data.get("PatientAge", "Unknown")
> slice_thick = float(dicom_data.get("SliceThickness", 0))
> ```


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║           DIFFERENT METHODS TO ACCESS TAGS                     ║
╚════════════════════════════════════════════════════════════════╝
""")

# Method 1: Direct attribute access
method1 = dicom_data.Modality
print(f"Method 1 - Attribute access: {method1}")

# Method 2: Using tag numbers
method2 = dicom_data[0x0008, 0x0060].value
print(f"Method 2 - Tag number: {method2}")

# Method 3: Using get() method (safer - won't crash if tag doesn't exist)
method3 = dicom_data.get("Modality", "Not found")
print(f"Method 3 - get() method: {method3}")

# Method 4: Using tag name as string
method4 = dicom_data.data_element("Modality").value
print(f"Method 4 - data_element(): {method4}")


## Ενότητα 8 — Πίνακας Σύνοψης Tags με `pandas`

### Γιατί DataFrame;

Όταν δουλεύετε με **πολλά** tags, δεν θέλετε `print(...)` παντού. Θέλετε **έναν πίνακα** που να μπορείτε να:
- Φιλτράρετε,
- Ταξινομήσετε,
- Εξάγετε σε CSV/Excel,
- Συγκρίνετε με άλλο DICOM.

Το `pandas.DataFrame` είναι το εργαλείο της Python για ακριβώς αυτή τη δουλειά — μια **2D δομή με ονόματα στηλών**, σαν Excel αλλά προγραμματιζόμενο.

### Τι κάνει η συνάρτηση `create_dicom_summary`

Δέχεται:
- `dicom_obj` — το DICOM που θέλουμε να εξετάσουμε.
- `tags_of_interest` — λίστα ονομάτων tag (προαιρετικό· έχει default).

Επιστρέφει DataFrame με τρεις στήλες:

| Tag Name | Tag Number | Value |
|----------|------------|-------|
| PatientName | (0010, 0010) | CompressedSamples^CT1 |
| Modality | (0008, 0060) | CT |
| Rows | (0028, 0010) | 128 |
| ... | ... | ... |

### Παιδαγωγικά σημεία στον κώδικα

#### 1. Default arguments
```python
def create_dicom_summary(dicom_obj, tags_of_interest=None):
    if tags_of_interest is None:
        tags_of_interest = [...]
```
Το `None` ως default μας επιτρέπει να φτιάξουμε **διαφορετική προεπιλεγμένη λίστα κάθε φορά**. (Αν είχαμε `tags_of_interest=[]` ως default, θα μοιραζόμασταν την ίδια λίστα μεταξύ κλήσεων — γνωστό «mutable default argument» bug της Python!)

#### 2. Defensive coding με `try/except`
Μέσα στο loop, αν κάποιο tag δεν υπάρχει, καταγράφουμε «Not available» αντί να κρασάρουμε. Σε **πραγματικό** σενάριο που σαρώνετε χιλιάδες DICOM, αυτό είναι αναγκαίο.

#### 3. Επαναχρησιμοποιήσιμη συνάρτηση
Δεν είναι hardcoded για ένα συγκεκριμένο αρχείο. Μπορείτε να την καλέσετε σε **οποιοδήποτε** DICOM, με **οποιαδήποτε** λίστα tags. Αυτό λέγεται **abstraction** — βασική αρχή του καλού κώδικα.

> 💡 **Επέκταση για άσκηση:** Φτιάξτε μια συνάρτηση `compare_dicoms(dicom1, dicom2)` που επιστρέφει DataFrame με τις τιμές δίπλα-δίπλα και χρωματίζει τις διαφορές. Ιδανικό για QC σε πειραματικά πρωτόκολλα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              CREATING A TAGS SUMMARY TABLE                     ║
╚════════════════════════════════════════════════════════════════╝
""")

def create_dicom_summary(dicom_obj, tags_of_interest=None):
    """
    Create a pandas DataFrame summarizing important DICOM tags
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        The DICOM object to summarize
    tags_of_interest : list
        List of tag names to include. If None, uses default important tags.
    
    Returns:
    --------
    pd.DataFrame : Summary of DICOM tags
    """
    
    if tags_of_interest is None:
        tags_of_interest = [
            'PatientName', 'PatientID', 'PatientSex', 'PatientBirthDate',
            'StudyDate', 'StudyDescription', 'Modality',
            'Manufacturer', 'ManufacturerModelName',
            'Rows', 'Columns', 'PixelSpacing', 'SliceThickness',
            'ImagePositionPatient', 'ImageOrientationPatient'
        ]
    
    data = []
    for tag_name in tags_of_interest:
        try:
            value = getattr(dicom_obj, tag_name)
            # Get the tag number
            tag = dicom_obj.data_element(tag_name).tag
            data.append({
                'Tag Name': tag_name,
                'Tag Number': str(tag),
                'Value': str(value)
            })
        except AttributeError:
            data.append({
                'Tag Name': tag_name,
                'Tag Number': 'N/A',
                'Value': 'Not available'
            })
    
    return pd.DataFrame(data)

# Create and display the summary
summary_df = create_dicom_summary(dicom_data)
display(summary_df)


## Ενότητα 9 — Πρόσβαση στα Δεδομένα της Εικόνας

### Από metadata σε pixels

Μέχρι τώρα είδαμε μόνο **κείμενο και αριθμούς** — metadata. Σε αυτή την ενότητα ζητάμε για πρώτη φορά τα **ίδια τα pixel** της εικόνας με την ιδιότητα:

```python
pixel_array = dicom_data.pixel_array
```

### Τι είναι το `pixel_array`;

Είναι ένας **NumPy ndarray** — δηλαδή ένας πολυδιάστατος πίνακας αριθμών. Σε μια απλή 2D DICOM εικόνα η μορφή του είναι `(Rows, Columns)`. Σε μια έγχρωμη ή πολυτομική εικόνα μπορεί να έχει 3 ή 4 διαστάσεις.

### Οι πέντε ερωτήσεις πρώτης γραμμής

Πάντα, σε **κάθε** εικόνα που σας δίνει κάποιος, ξεκινήστε με αυτές τις πέντε ερωτήσεις:

| Ερώτηση | Κώδικας | Γιατί έχει σημασία |
|---------|---------|----------------------|
| Πόσο μεγάλη είναι; | `.shape` | Επιβεβαίωση δομής, υπολογισμός μνήμης |
| Τι τύπος δεδομένων; | `.dtype` | int16, uint16, float32 — επηρεάζει εύρος |
| Ελάχιστη τιμή; | `.min()` | Έλεγχος εγκυρότητας |
| Μέγιστη τιμή; | `.max()` | Δυναμικό εύρος |
| Μέση τιμή; | `.mean()` | «Φωτεινότητα» |

> 🧠 **Παράδειγμα ερμηνείας:** Αν δείτε CT εικόνα με `min=0, max=4095`, **δεν** είναι ακόμα σε Hounsfield Units. Αν δείτε `min=-1024, max=3071`, είναι ήδη σε HU. Αυτή η μικρή λεπτομέρεια είναι κρίσιμη — θα τη συζητήσουμε αναλυτικά στην Ενότητα 11.

### Συνηθισμένοι τύποι δεδομένων στις ιατρικές εικόνες

| dtype | Εύρος | Πού συναντάται |
|-------|-------|------------------|
| `uint8` | 0–255 | Πιο σπάνιο, μερικά ultrasound |
| `uint16` | 0–65535 | CT, MRI πριν το rescale |
| `int16` | −32768–32767 | CT μετά το rescale (Hounsfield) |
| `float32` | μεγάλο εύρος | Quantitative MRI, parametric maps |

> ⚠️ **Προσοχή με τους ακέραιους:** Αν κάνετε πράξεις σε `uint16` και υπάρξει υπερχείλιση (overflow) ή αρνητικό αποτέλεσμα, παίρνετε εντελώς λάθος νούμερα **σιωπηλά**. Πάντα μετατρέπετε σε `float` πριν από αριθμητικές πράξεις: `img = pixel_array.astype(np.float32)`.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                 ACCESSING IMAGE DATA                           ║
╚════════════════════════════════════════════════════════════════╝
""")

# Get the pixel array
pixel_array = dicom_data.pixel_array

print(f"Image shape: {pixel_array.shape}")
print(f"Data type: {pixel_array.dtype}")
print(f"Min pixel value: {pixel_array.min()}")
print(f"Max pixel value: {pixel_array.max()}")
print(f"Mean pixel value: {pixel_array.mean():.2f}")
print(f"Standard deviation: {pixel_array.std():.2f}")


## Ενότητα 10 — Οπτικοποίηση Εικόνων DICOM

### Από αριθμούς σε εικόνα

Μέχρι τώρα οι «εικόνες» μας ήταν πίνακες αριθμών. Τώρα τους **βλέπουμε** ως πραγματικές εικόνες με τη `matplotlib`.

### Η `imshow` — η συνάρτηση-κλειδί

```python
plt.imshow(pixel_array, cmap='gray')
```

Δύο σημαντικές παράμετροι:

#### 1. `cmap='gray'` — η χρωματική κλίμακα
Η matplotlib από προεπιλογή χρησιμοποιεί `viridis` (μπλε-κίτρινο), που είναι ωραίο για επιστημονικά γραφήματα **αλλά ΟΧΙ για ιατρικές εικόνες**. Οι ακτινολόγοι αναγνωρίζουν παθολογία σε γκρι κλίμακα — βλέποντας μια CT σε ψυχρά-θερμά χρώματα μπορεί να μη δουν αυτό που ψάχνουν.

#### 2. Auto-scaling vs manual range
Η `imshow` από προεπιλογή κάνει **αυτόματη κανονικοποίηση**: αντιστοιχίζει τη min τιμή στο μαύρο και τη max στο λευκό. Αυτό είναι:
- **Καλό** για γρήγορη εξερεύνηση.
- **Κακό** όταν θέλετε σωστή σύγκριση μεταξύ εικόνων ή σωστή HU απεικόνιση. Σε αυτές τις περιπτώσεις προτιμούμε **windowing** (Ενότητα 12).

### Συμπληρωματικά χαρακτηριστικά

Η συνάρτηση `display_dicom_image` προσθέτει:

- **`colorbar`** — μπάρα δεξιά που δείχνει τι τιμή είναι το λευκό/μαύρο. Σημαντικό!
- **`title`** με modality και διαστάσεις.
- **`xlabel`/`ylabel`** — άξονες με νόημα (αν και σε ιατρικές εικόνες συχνά τους κρύβουμε με `axis('off')`).
- **`text`** σε γωνία — info box με min/max/mean.

### Παιδαγωγικά σημεία

> 💡 **Πάντα να βάζετε colorbar.** Χωρίς αυτή, ο θεατής δεν ξέρει αν το «λευκό» αντιπροσωπεύει 100 ή 10000. Στην ιατρική απεικόνιση, αυτό κάνει τη διαφορά.

> 🎯 **`figsize=(10, 10)`** — τετράγωνο μέγεθος. Οι ιατρικές εικόνες είναι συνήθως τετράγωνες (π.χ. 512×512). Αν βάλετε ορθογώνιο figsize, παραμορφώνεται οπτικά.

> ⚠️ **Coordinate systems:** Στην matplotlib το pixel `(0,0)` είναι **πάνω-αριστερά** της εικόνας. Στις ιατρικές εικόνες, οι συντεταγμένες ασθενούς (LPS/RAS) είναι διαφορετικές. Όταν κάνετε segmentation και θέλετε να αναφερθείτε σε «δεξί πνεύμονα», βεβαιωθείτε ποια κατεύθυνση είναι ποια!


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              VISUALIZING DICOM IMAGES                          ║
╚════════════════════════════════════════════════════════════════╝
""")

def display_dicom_image(dicom_obj, title="DICOM Image", figsize=(10, 10)):
    """
    Display a DICOM image with proper window/level settings
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        The DICOM object to display
    title : str
        Title for the plot
    figsize : tuple
        Figure size
    """
    
    plt.figure(figsize=figsize)
    
    # Get pixel array
    pixel_array = dicom_obj.pixel_array
    
    # Display the image
    plt.imshow(pixel_array, cmap='gray')
    plt.colorbar(label='Pixel Intensity')
    plt.title(f"{title}\n{dicom_obj.Modality} - {dicom_obj.Rows}x{dicom_obj.Columns}")
    plt.xlabel('Column (X)')
    plt.ylabel('Row (Y)')
    
    # Add image information
    info_text = f"Min: {pixel_array.min()}, Max: {pixel_array.max()}, Mean: {pixel_array.mean():.1f}"
    plt.text(0.02, 0.98, info_text, transform=plt.gca().transAxes,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
             verticalalignment='top', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Display the image
display_dicom_image(dicom_data, "Sample DICOM CT Image")


## Ενότητα 11 — Hounsfield Units (αποκλειστικά για CT)

### Τι είναι οι Hounsfield Units;

Στην **Υπολογιστική Τομογραφία (CT)**, οι τιμές των pixel **δεν** είναι αυθαίρετοι αριθμοί — αντιπροσωπεύουν την **εξασθένιση των ακτίνων Χ** σε σχέση με το νερό. Αυτή η κανονικοποιημένη κλίμακα λέγεται **Hounsfield Units** (HU), από τον Sir Godfrey Hounsfield (Νόμπελ Φυσιολογίας 1979).

Ο ορισμός:
$$
\text{HU} = 1000 \times \dfrac{\mu_{ιστού} - \mu_{νερό}}{\mu_{νερό} - \mu_{αέρα}}
$$

όπου **μ** ο γραμμικός συντελεστής εξασθένισης.

### Η αξία της κλίμακας: συγκρισιμότητα

Επειδή η κλίμακα HU είναι **απόλυτη και τυποποιημένη**, ένας ιστός με συγκεκριμένη σύσταση δίνει την **ίδια** τιμή σε οποιοδήποτε CT μηχάνημα στον κόσμο. Αυτό είναι κρίσιμο.

### Πίνακας τιμών HU για βασικούς ιστούς

| Ιστός / υλικό | HU |
|----------------|-----|
| Αέρας | −1000 |
| Πνεύμονας | −500 έως −400 |
| Λίπος | −100 έως −50 |
| Νερό | 0 |
| Μυς | 30 έως 50 |
| Αίμα (φρέσκο) | 50 έως 80 |
| Συκώτι | 50 έως 70 |
| Εγκέφαλος (φαιά ουσία) | 35 έως 45 |
| Σπογγώδες οστό | 200 έως 700 |
| Συμπαγές οστό | 1000+ |
| Μέταλλο (εμφυτεύματα) | 2000+ (συχνά «γεμίζει» με streak artifacts) |

### Η μετατροπή — γιατί χρειάζεται;

Το DICOM **δεν αποθηκεύει** πάντα τις τιμές HU απευθείας. Για λόγους αποδοτικότητας, αποθηκεύει «raw» τιμές pixel και δίνει δύο tags για τη μετατροπή:

- **`RescaleSlope`** (συνήθως 1)
- **`RescaleIntercept`** (συνήθως −1024 για CT)

Η μετατροπή είναι γραμμική:

$$
\text{HU} = (\text{pixel\_value} \times \text{RescaleSlope}) + \text{RescaleIntercept}
$$

### Τι κάνει η συνάρτηση `convert_to_hounsfield`

1. Παίρνει τα pixel.
2. Τα μετατρέπει σε `float64` για ακρίβεια.
3. Διαβάζει `RescaleSlope` και `RescaleIntercept` από τα tags (με **defaults** 1 και 0 αν λείπουν).
4. Εφαρμόζει τη γραμμική μετατροπή.
5. Επιστρέφει τη **σωστή** εικόνα HU.

> ⚠️ **ΠΡΟΣΟΧΗ — μην εφαρμόσετε σε non-CT:** Η μετατροπή HU **έχει νόημα μόνο σε CT**. Σε MRI, οι τιμές είναι **σχετικές** (δεν υπάρχει «φυσική κλίμακα»). Σε υπερηχογράφημα, ομοίως. Γι' αυτό ο κώδικας ελέγχει `if dicom_data.Modality == 'CT'` πριν τρέξει τη μετατροπή.

> 🎯 **Πρακτική σύσταση:** Στα CT projects σας, **πάντα** μετατρέπετε σε HU πριν κάνετε οτιδήποτε άλλο. Είναι το πρώτο preprocessing βήμα για segmentation, radiomics, ML κ.λπ.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║           HOUNSFIELD UNITS (for CT images)                     ║
╚════════════════════════════════════════════════════════════════╝

For CT images, pixel values are in Hounsfield Units (HU):
- Air: -1000 HU
- Water: 0 HU
- Soft tissue: 40-80 HU
- Bone: 400-1000 HU

DICOM stores pixel values that need to be converted using:
HU = pixel_value * RescaleSlope + RescaleIntercept
""")

def convert_to_hounsfield(dicom_obj):
    """
    Convert pixel values to Hounsfield Units
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object
    
    Returns:
    --------
    np.ndarray : Image in Hounsfield Units
    """
    
    pixel_array = dicom_obj.pixel_array.astype(np.float64)
    
    # Get rescale parameters
    intercept = dicom_obj.RescaleIntercept if hasattr(dicom_obj, 'RescaleIntercept') else 0
    slope = dicom_obj.RescaleSlope if hasattr(dicom_obj, 'RescaleSlope') else 1
    
    # Convert to HU
    hu_image = pixel_array * slope + intercept
    
    print(f"Rescale Slope: {slope}")
    print(f"Rescale Intercept: {intercept}")
    print(f"HU range: [{hu_image.min():.1f}, {hu_image.max():.1f}]")
    
    return hu_image

# Convert to Hounsfield Units if it's a CT image
if dicom_data.Modality == 'CT':
    hu_image = convert_to_hounsfield(dicom_data)
else:
    print("This is not a CT image, Hounsfield conversion not applicable.")


## Ενότητα 12 — Windowing (Window/Level)

### Το πρόβλημα του δυναμικού εύρους

Η ανθρώπινη όραση **ξεχωρίζει περίπου 100 αποχρώσεις του γκρι**. Όμως μια CT εικόνα μπορεί να περιέχει τιμές από −1000 (αέρας) έως +3000 HU (οστό) — **4000 διαφορετικές τιμές**. Αν τις απεικονίσουμε όλες ταυτόχρονα στις 256 αποχρώσεις του γκρι μιας οθόνης, **χάνουμε τη λεπτομέρεια** σε κάθε ιστό.

**Windowing:** μια τεχνική που λέει «δείξε μου με αντίθεση **μόνο** ένα συγκεκριμένο εύρος HU, και κόψε τα υπόλοιπα στο μαύρο/λευκό».

### Δύο παράμετροι

| Παράμετρος | Σύμβολο | Σημασία |
|------------|---------|---------|
| **Window Center** (Level) | C ή WL | Η μέση τιμή HU που μας ενδιαφέρει |
| **Window Width** | W ή WW | Πόσο πλατύ εύρος γύρω από το center |

Το εύρος εμφάνισης είναι:
$$
[C - W/2, \quad C + W/2]
$$

- Pixel με τιμή **κάτω** από `C - W/2` → μαύρο.
- Pixel με τιμή **πάνω** από `C + W/2` → λευκό.
- Ενδιάμεσες τιμές → γραμμική κλιμάκωση στο γκρι.

### Συνηθισμένα presets για CT

| Όνομα | Center (HU) | Width (HU) | Τι αναδεικνύει |
|-------|-------------|------------|------------------|
| **Lung** | −600 | 1500 | Παρέγχυμα πνεύμονα, οζίδια |
| **Mediastinum / Soft tissue** | 50 | 350–400 | Ήπαρ, νεφροί, καρδιά |
| **Bone** | 400 | 1800 | Σπονδυλική στήλη, κατάγματα |
| **Brain** | 40 | 80 | Λεπτή αντίθεση φαιάς-λευκής ουσίας |
| **Stroke / acute brain** | 35 | 30 | Πρώιμα σημεία ισχαιμίας |
| **Liver** | 60 | 160 | Εστιακές βλάβες ήπατος |

> 💡 **Στην κλινική πράξη:** Ο ακτινολόγος αλλάζει συνεχώς windows κατά την ερμηνεία. Δεν βλέπει το CT με ένα μόνο preset — αλλάζει για να ελέγξει διαφορετικές δομές.

### Τι κάνει η συνάρτηση `apply_window`

```python
def apply_window(image, window_center, window_width):
    img_min = window_center - window_width // 2
    img_max = window_center + window_width // 2
    windowed = np.clip(image, img_min, img_max)         # κόβει στα όρια
    windowed = ((windowed - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)
    return windowed
```

Τρία βήματα:
1. **`np.clip`** — όλες οι τιμές κάτω από `img_min` γίνονται `img_min`, πάνω από `img_max` γίνονται `img_max`.
2. **Κανονικοποίηση** στο εύρος [0, 1].
3. **Κλιμάκωση** σε [0, 255] και μετατροπή σε `uint8` για εμφάνιση.

### Σημαντική διάκριση

> 🎯 **Windowing = ρύθμιση εμφάνισης ΜΟΝΟ.** ΔΕΝ αλλάζει τα δεδομένα της εικόνας. Αν θέλετε να κάνετε ποσοτική ανάλυση (π.χ. μετρήσετε τη μέση HU ενός όγκου), χρησιμοποιείτε την **αρχική** εικόνα HU, όχι τη windowed. Το windowed είναι μόνο για να **βλέπετε**.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  WINDOWING (Window/Level)                      ║
╚════════════════════════════════════════════════════════════════╝

Windowing is used to enhance contrast in specific tissue ranges.
- Window Center (Level): middle of the range of HU values to display
- Window Width: range of HU values to display

Common CT windows:
- Lung: Center=-600, Width=1500
- Mediastinum: Center=50, Width=350
- Bone: Center=400, Width=1800
- Brain: Center=40, Width=80
""")

def apply_window(image, window_center, window_width):
    """
    Apply windowing to an image
    
    Parameters:
    -----------
    image : np.ndarray
        Input image (typically in HU for CT)
    window_center : float
        Window center (level)
    window_width : float
        Window width
    
    Returns:
    --------
    np.ndarray : Windowed image scaled to 0-255
    """
    
    img_min = window_center - window_width // 2
    img_max = window_center + window_width // 2
    
    windowed = np.clip(image, img_min, img_max)
    windowed = ((windowed - img_min) / (img_max - img_min) * 255.0).astype(np.uint8)
    
    return windowed

def display_multiple_windows(dicom_obj):
    """
    Display the same CT image with different window settings
    """
    
    if dicom_obj.Modality != 'CT':
        print("Windowing demonstration is most relevant for CT images.")
        return
    
    hu_image = convert_to_hounsfield(dicom_obj)
    
    # Define different window settings
    windows = {
        'Soft Tissue': (40, 400),
        'Lung': (-600, 1500),
        'Bone': (400, 1800),
        'Brain': (40, 80)
    }
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 15))
    axes = axes.ravel()
    
    for idx, (name, (center, width)) in enumerate(windows.items()):
        windowed = apply_window(hu_image, center, width)
        axes[idx].imshow(windowed, cmap='gray')
        axes[idx].set_title(f'{name} Window\nCenter: {center}, Width: {width}')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Display with different windows if CT
if dicom_data.Modality == 'CT':
    display_multiple_windows(dicom_data)


## Ενότητα 13 — Φόρτωση Πολλών Αρχείων (Series)

### Τι είναι μια «σειρά» (series);

Σπάνια μια ιατρική εξέταση είναι μία μόνο εικόνα. Πιο συχνά είναι μια **σειρά τομών** (slices) που μαζί συνθέτουν τρισδιάστατη πληροφορία:

- **CT θώρακος**: 200–500 αξονικές τομές, μία ανά ~1mm.
- **MRI εγκεφάλου**: 30–200 τομές ανά ακολουθία, και πολλές ακολουθίες (T1, T2, FLAIR, DWI...).
- **DCE-MRI**: τομές × χρονικές στιγμές → 4D δεδομένα.

Στο DICOM, **κάθε τομή είναι ξεχωριστό αρχείο**. Μια σειρά είναι ένας **φάκελος** με δεκάδες ή εκατοντάδες αρχεία.

### Η ιεραρχία DICOM

```
Patient
└── Study  (μια εξέταση, π.χ. «MRI εγκεφάλου 15/03/2024»)
    ├── Series 1: T1
    │   ├── Image 1 (slice 1)
    │   ├── Image 2 (slice 2)
    │   └── ...
    ├── Series 2: T2
    └── Series 3: FLAIR
```

Κάθε επίπεδο έχει το δικό του **UID**: `PatientID`, `StudyInstanceUID`, `SeriesInstanceUID`, `SOPInstanceUID`.

### Το κρίσιμο πρόβλημα: η σειρά των τομών

Όταν διαβάζετε αρχεία από έναν φάκελο, τα παίρνετε **σε αλφαβητική σειρά** του filename. Αυτή **σπανίως** είναι η σωστή ανατομική σειρά! Παράδειγμα: `IM_0001.dcm, IM_0010.dcm, IM_0011.dcm, ..., IM_0002.dcm` — αλφαβητικά αλλά όχι σωστά.

### Δύο τρόποι σωστής ταξινόμησης

#### 1. `InstanceNumber`
Ένας ακέραιος που δίνει το μηχάνημα. Συνήθως αξιόπιστο.

```python
dicom_files.sort(key=lambda x: int(x.InstanceNumber))
```

#### 2. `ImagePositionPatient[2]` (z-coordinate)
Η **πραγματική** ανατομική θέση στον άξονα z (κρανιοκαυδικός). Πιο αξιόπιστο όταν το `InstanceNumber` είναι αναξιόπιστο.

```python
dicom_files.sort(key=lambda x: float(x.ImagePositionPatient[2]))
```

### Τι κάνει η `load_dicom_series`

1. Διασχίζει τον φάκελο με `os.listdir`.
2. Προσπαθεί να φορτώσει κάθε αρχείο. Αν αποτύχει (π.χ. δεν είναι DICOM), το προσπερνάει.
3. Ταξινομεί κατά `InstanceNumber`.
4. Επιστρέφει τη λίστα.

> 💡 **Πιο εύρωστο:** Χρησιμοποιήστε `pydicom.dcmread(filepath, stop_before_pixels=True)` αρχικά για ταχύτητα, ταξινομήστε, και μόνο μετά διαβάστε τα pixel των αρχείων που σας ενδιαφέρουν.

### Τι κάνει το demo

Αντί για πραγματικό φάκελο, ο κώδικας φορτώνει **τρία διαφορετικά** built-in test files (`CT_small.dcm`, `MR_small.dcm`, `rtplan.dcm`) για να έχετε υλικό για επόμενες ενότητες (montage, viewer). Αυτά **δεν** ανήκουν στην ίδια σειρά — απλά είναι demos.

> 🧠 **Σημείωση:** Σε πραγματική σειρά, όλα τα αρχεία θα είχαν τον **ίδιο** `SeriesInstanceUID`. Αυτό είναι ο σίγουρος τρόπος να επιβεβαιώσετε ότι ανήκουν μαζί.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          LOADING MULTIPLE DICOM FILES (SERIES)                 ║
╚════════════════════════════════════════════════════════════════╝

Medical imaging studies often consist of multiple slices (series).
Let's learn how to load and navigate through them.
""")

def load_dicom_series(directory_path):
    """
    Load all DICOM files from a directory and sort them by instance number
    
    Parameters:
    -----------
    directory_path : str
        Path to directory containing DICOM files
    
    Returns:
    --------
    list : List of sorted DICOM objects
    """
    
    dicom_files = []
    
    # Get all files in directory
    for filename in os.listdir(directory_path):
        filepath = os.path.join(directory_path, filename)
        try:
            dcm = pydicom.dcmread(filepath)
            dicom_files.append(dcm)
        except:
            continue  # Skip non-DICOM files
    
    # Sort by Instance Number if available
    try:
        dicom_files.sort(key=lambda x: int(x.InstanceNumber))
    except:
        print("Warning: Could not sort by InstanceNumber")
    
    print(f"Loaded {len(dicom_files)} DICOM files")
    return dicom_files

# For demonstration, we'll load multiple sample files
print("Loading sample DICOM series...")

# Get multiple sample files for demonstration
sample_files = []
try:
    # Try to load multiple sample files
    sample_paths = ['CT_small.dcm', 'MR_small.dcm', 'rtplan.dcm']
    for path_name in sample_paths:
        try:
            sample_files.append(pydicom.dcmread(get_testdata_file(path_name)))
        except:
            pass
    print(f"Loaded {len(sample_files)} sample files for demonstration")
except Exception as e:
    print(f"Note: {e}")
    sample_files = [dicom_data]  # Use our original file


## Ενότητα 14 — Διαδραστικός Slice Viewer (κλάση `DICOMSliceViewer`)

### Γιατί κλάση και όχι απλές συναρτήσεις;

Μέχρι τώρα γράφαμε **συναρτήσεις**: εργαλεία που παίρνουν δεδομένα και επιστρέφουν αποτελέσματα. Όταν όμως έχετε:
- Δεδομένα που θέλετε να **κρατήσετε** (η σειρά τομών),
- Πολλές διαφορετικές λειτουργίες πάνω στα ίδια δεδομένα (show, montage, 3D, orthogonal views),
- Κατάσταση που αλλάζει (ποια τομή βλέπω τώρα),

τότε η **κλάση** είναι η σωστή δομή. Μια κλάση «δένει» δεδομένα και λειτουργίες σε ένα αντικείμενο.

### Η ανατομία της κλάσης

```python
class DICOMSliceViewer:
    def __init__(self, dicom_series):  # constructor
        self.dicom_series = dicom_series
        self.current_slice = 0
        self.num_slices = len(dicom_series)
    
    def show_slice(self, slice_idx): ...
    def show_montage(self, num_cols=4): ...
    def create_3d_volume(self): ...
    def show_orthogonal_views(self): ...
```

- **`__init__`**: «κατασκευαστής». Καλείται όταν φτιάχνετε αντικείμενο: `viewer = DICOMSliceViewer(files)`. Αποθηκεύει τα δεδομένα ως **attributes** (`self.dicom_series`).
- **`self`**: αναφέρεται στο τρέχον αντικείμενο. Είναι το πρώτο όρισμα κάθε method.
- **Methods**: συναρτήσεις-μέλη της κλάσης. Έχουν πρόσβαση στα attributes.

### Οι τέσσερις λειτουργίες

#### 1. `show_slice(slice_idx)` — μία τομή τη φορά
Ζωγραφίζει συγκεκριμένη τομή με matplotlib και δείχνει info (instance number, position).

#### 2. `show_montage(num_cols=4)` — όλες οι τομές σε grid
Όλες μαζί σε πλέγμα. Χρήσιμο για **γρήγορη επισκόπηση** ολόκληρης της σειράς πριν εμβαθύνετε.

#### 3. `create_3d_volume()` — από 2D σε 3D
Στοιβάζει τις τομές σε **3D NumPy array** σχήματος `(slices, rows, columns)`.
```python
volume = np.stack([dcm.pixel_array for dcm in self.dicom_series], axis=0)
```
Αυτή είναι η βάση για **ογκομετρική ανάλυση**: segmentation, registration, MIP κ.λπ.

#### 4. `show_orthogonal_views()` — οι τρεις ανατομικές προβολές

Από τον 3D όγκο εξάγουμε τρεις προβολές μέσα από τη μέση:

| Προβολή | Slicing | Τι δείχνει |
|---------|---------|------------|
| **Axial** (εγκάρσια) | `volume[mid_z, :, :]` | Όπως κοιτάζουμε από τα πόδια προς το κεφάλι |
| **Coronal** (στεφανιαία) | `volume[:, mid_y, :]` | Από εμπρός προς πίσω |
| **Sagittal** (οβελιαία) | `volume[:, :, mid_x]` | Από δεξιά προς αριστερά |

> 🎯 **Σημαντικό:** Στις coronal και sagittal προβολές, χρειάζεστε `aspect='auto'` ή ρητή κλιμάκωση γιατί συχνά το **slice spacing** διαφέρει από το **pixel spacing** — αν δεν διορθώσετε, η εικόνα θα φαίνεται «πατικωμένη» ή «τραβηγμένη».

### Παιδαγωγικά σημεία

> 💡 **OOP για αρχάριους:** Η κλάση είναι σαν «καλούπι». Φτιάχνετε αντικείμενα από αυτή. Κάθε αντικείμενο έχει τα δικά του δεδομένα αλλά τις ίδιες ικανότητες.

> 🧠 **Επόμενο βήμα:** Δοκιμάστε να προσθέσετε δικό σας method `apply_window_to_all(center, width)` που εφαρμόζει windowing σε όλες τις τομές της σειράς. Έτσι θα νιώσετε τη δύναμη του OOP.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              INTERACTIVE SLICE VIEWER                          ║
╚════════════════════════════════════════════════════════════════╝
""")

class DICOMSliceViewer:
    """
    Interactive viewer for navigating through DICOM slices
    """
    
    def __init__(self, dicom_series):
        """
        Initialize viewer with a series of DICOM files
        
        Parameters:
        -----------
        dicom_series : list
            List of DICOM objects
        """
        self.dicom_series = dicom_series
        self.current_slice = 0
        self.num_slices = len(dicom_series)
        
    def show_slice(self, slice_idx):
        """Display a specific slice"""
        
        if slice_idx < 0 or slice_idx >= self.num_slices:
            print(f"Slice index out of range. Valid range: 0-{self.num_slices-1}")
            return
        
        self.current_slice = slice_idx
        dcm = self.dicom_series[slice_idx]
        
        plt.figure(figsize=(10, 10))
        plt.imshow(dcm.pixel_array, cmap='gray')
        plt.title(f'Slice {slice_idx + 1}/{self.num_slices}\n'
                  f'Modality: {dcm.Modality}')
        plt.colorbar(label='Pixel Intensity')
        plt.axis('on')
        plt.tight_layout()
        plt.show()
        
        # Display slice information
        print(f"\n📊 Slice {slice_idx + 1} Information:")
        print(f"   Instance Number: {dcm.get('InstanceNumber', 'N/A')}")
        print(f"   Image Position: {dcm.get('ImagePositionPatient', 'N/A')}")
        print(f"   Shape: {dcm.pixel_array.shape}")
    
    def show_montage(self, num_cols=4):
        """Display all slices in a montage"""
        
        num_slices = len(self.dicom_series)
        num_rows = (num_slices + num_cols - 1) // num_cols
        
        fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, 4*num_rows))
        
        if num_slices == 1:
            axes = np.array([axes])
        axes = axes.ravel()
        
        for idx in range(num_cols * num_rows):
            if idx < num_slices:
                axes[idx].imshow(self.dicom_series[idx].pixel_array, cmap='gray')
                axes[idx].set_title(f'Slice {idx + 1}', fontsize=10)
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    def create_3d_volume(self):
        """Stack slices to create a 3D volume"""
        
        if self.num_slices < 2:
            print("Need at least 2 slices to create a volume")
            return None
        
        # Stack all slices
        volume = np.stack([dcm.pixel_array for dcm in self.dicom_series], axis=0)
        
        print(f"3D Volume shape: {volume.shape}")
        print(f"(slices, rows, columns)")
        
        return volume
    
    def show_orthogonal_views(self):
        """Show axial, coronal, and sagittal views"""
        
        volume = self.create_3d_volume()
        
        if volume is None:
            return
        
        # Get middle slices
        mid_slice = volume.shape[0] // 2
        mid_row = volume.shape[1] // 2
        mid_col = volume.shape[2] // 2
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Axial view
        axes[0].imshow(volume[mid_slice, :, :], cmap='gray')
        axes[0].set_title('Axial View')
        axes[0].axis('off')
        
        # Coronal view
        axes[1].imshow(volume[:, mid_row, :], cmap='gray', aspect='auto')
        axes[1].set_title('Coronal View')
        axes[1].axis('off')
        
        # Sagittal view
        axes[2].imshow(volume[:, :, mid_col], cmap='gray', aspect='auto')
        axes[2].set_title('Sagittal View')
        axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()

# Create viewer instance
viewer = DICOMSliceViewer(sample_files)

print(f"\n✓ Viewer created with {viewer.num_slices} slice(s)")

# Show first slice
viewer.show_slice(0)

# Show montage if multiple slices
if len(sample_files) > 1:
    viewer.show_montage()


## Ενότητα 15 — Εξαγωγή Πίνακα Metadata από Σειρά

### Από τομές σε πίνακα-σύνοψη

Όταν δουλεύετε με σειρά **πολλών** τομών, χρειάζεστε γρήγορη επισκόπηση: «τι περιέχει αυτό το dataset;». Η συνάρτηση `extract_series_metadata` φτιάχνει ένα DataFrame με μία γραμμή ανά τομή, ώστε να μπορείτε να ελέγξετε σε ένα ξεκάθαρο πίνακα:

| Slice | Instance Number | Modality | Rows | Columns | Slice Thickness | Image Position |
|-------|-----------------|----------|------|---------|------------------|------------------|

### Πότε χρησιμεύει στην πράξη

- **Quality control:** Όλες οι τομές έχουν την ίδια χωρική ανάλυση; Έχουν ίδιο πάχος; Είναι ίδιες διαστάσεις;
- **Πλοήγηση:** Σε ποιες τομές βλέπω τη συγκεκριμένη ανατομική περιοχή;
- **Διαλογή:** Αν η σειρά έχει «παραπανίσιες» τομές από scout/localizer, τις εντοπίζω και τις αφαιρώ.

### Το «είδος» κώδικα: list comprehension με dict

Η συνάρτηση συσσωρεύει μία εγγραφή ανά τομή:

```python
metadata.append({
    'Slice': idx + 1,
    'Instance Number': dcm.get('InstanceNumber', 'N/A'),
    ...
})
return pd.DataFrame(metadata)
```

Αυτό το μοτίβο **«λίστα από dictionaries → DataFrame»** είναι ο πιο καθαρός τρόπος να φτιάξετε DataFrame όταν τα δεδομένα έρχονται γραμμή-γραμμή.

### Παρατηρήστε την defensive πρόσβαση

Παντού `dcm.get('TagName', 'N/A')` αντί για `dcm.TagName`. Σε πραγματικές σειρές, θα συναντήσετε τομές που λείπουν tags (ειδικά σε ανωνυμοποιημένα ή παλιά δεδομένα). Με την `get()`, η συνάρτηση **δουλεύει** ακόμη και τότε.

> 💡 **Επέκταση για άσκηση:** Προσθέστε στήλες για `KVP`, `XRayTubeCurrent` (CT) ή `RepetitionTime`, `EchoTime` (MRI). Αυτά τα **acquisition parameters** είναι κρίσιμα για να ξέρετε τι ακριβώς σαρώθηκε.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║           EXTRACTING METADATA FROM SERIES                      ║
╚════════════════════════════════════════════════════════════════╝
""")

def extract_series_metadata(dicom_series):
    """
    Extract key metadata from a series of DICOM files
    
    Parameters:
    -----------
    dicom_series : list
        List of DICOM objects
    
    Returns:
    --------
    pd.DataFrame : Metadata table
    """
    
    metadata = []
    
    for idx, dcm in enumerate(dicom_series):
        metadata.append({
            'Slice': idx + 1,
            'Instance Number': dcm.get('InstanceNumber', 'N/A'),
            'Modality': dcm.get('Modality', 'N/A'),
            'Rows': dcm.Rows,
            'Columns': dcm.Columns,
            'Slice Thickness': dcm.get('SliceThickness', 'N/A'),
            'Image Position': str(dcm.get('ImagePositionPatient', 'N/A'))[:30] + '...'
        })
    
    return pd.DataFrame(metadata)

# Extract and display metadata
metadata_df = extract_series_metadata(sample_files)
display(metadata_df)


## Ενότητα 16 — Pixel Spacing & Πραγματικές Μετρήσεις

### Το θεμελιώδες ερώτημα

«Πόσα cm είναι ο όγκος;»
Δεν μπορείτε να απαντήσετε αν δεν ξέρετε **πόσα mm είναι κάθε pixel**.

### Τα κρίσιμα tags

| Tag | Σημασία | Μονάδες |
|-----|---------|---------|
| **`PixelSpacing`** | Απόσταση μεταξύ κέντρων pixel σε X και Y | mm |
| **`SliceThickness`** | Πάχος της κάθε τομής | mm |
| **`SpacingBetweenSlices`** | Απόσταση κέντρου-κέντρου μεταξύ διαδοχικών τομών | mm |
| **`ImagePositionPatient`** | Συντεταγμένες (x,y,z) του πρώτου pixel σε **patient coordinate system** | mm |
| **`ImageOrientationPatient`** | Διανύσματα row/column σε patient coords | unit vectors |

### Από pixel σε mm

Αν `PixelSpacing = [0.7, 0.7]`, τότε:

- Διάσταση εικόνας 512×512 pixels = **358.4 × 358.4 mm**.
- Ένας όγκος που εκτείνεται σε 50 pixels = **35 mm**.

### Voxel vs Pixel

- **Pixel** (picture element): 2D, στοιχείο εικόνας.
- **Voxel** (volume element): 3D, στοιχείο όγκου.

Σε μια 3D σειρά, ο όγκος ενός voxel είναι:
$$
V_{voxel} = \text{PixelSpacing}_x \times \text{PixelSpacing}_y \times \text{SliceThickness}
$$

Για να μετρήσετε π.χ. τον **όγκο όγκου**: μετράτε τα voxels στη segmentation και πολλαπλασιάζετε επί $V_{voxel}$.

### Anisotropic vs Isotropic δεδομένα

- **Isotropic**: ίδια ανάλυση και στις τρεις διαστάσεις (π.χ. 1×1×1 mm). Ιδανικά για 3D ανάλυση και deep learning.
- **Anisotropic**: διαφορετική ανάλυση (π.χ. 0.5×0.5×3 mm — συνηθισμένο σε MRI). Σε αυτή την περίπτωση, για 3D αλγορίθμους συχνά χρειάζεται **resampling** σε isotropic.

### Τι κάνει η `get_pixel_spacing_info`

1. Διαβάζει το `PixelSpacing`.
2. Υπολογίζει το **φυσικό μέγεθος** της εικόνας σε mm.
3. Διαβάζει το `SliceThickness`.
4. Τα τυπώνει.

### Συνηθισμένα λάθη στη μέτρηση

> ⚠️ **Λάθος #1:** Να μετρήσετε διάμετρο όγκου σε **pixels** και να την αναφέρετε σε αναφορά. Πάντα μετατρέπετε σε mm.

> ⚠️ **Λάθος #2:** Να ξεχάσετε ότι το spacing μπορεί να **διαφέρει** μεταξύ σειρών (T1 vs T2). Σε multi-modal ανάλυση, αυτό προκαλεί χάος αν δεν προσέξετε.

> ⚠️ **Λάθος #3:** Να αγνοήσετε το `ImageOrientationPatient`. Αν δύο σειρές έχουν διαφορετικό orientation και τις «στοιβάζετε» χωρίς registration, τα voxels δεν αντιστοιχούν στα ίδια ανατομικά σημεία!


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║        PIXEL SPACING & REAL-WORLD MEASUREMENTS                 ║
╚════════════════════════════════════════════════════════════════╝

Pixel spacing tells us the physical distance between pixels.
This is crucial for accurate measurements!
""")

def get_pixel_spacing_info(dicom_obj):
    """
    Extract pixel spacing information
    """
    
    print(f"\n📏 Spatial Information:")
    print("-" * 50)
    
    if hasattr(dicom_obj, 'PixelSpacing'):
        row_spacing, col_spacing = dicom_obj.PixelSpacing
        print(f"Pixel Spacing (Row, Column): {row_spacing} mm, {col_spacing} mm")
        
        # Calculate physical size of the image
        physical_width = dicom_obj.Columns * col_spacing
        physical_height = dicom_obj.Rows * row_spacing
        
        print(f"Image size in pixels: {dicom_obj.Columns} x {dicom_obj.Rows}")
        print(f"Image size in mm: {physical_width:.2f} x {physical_height:.2f}")
    else:
        print("Pixel spacing information not available")
    
    if hasattr(dicom_obj, 'SliceThickness'):
        print(f"Slice Thickness: {dicom_obj.SliceThickness} mm")

get_pixel_spacing_info(dicom_data)


## Ενότητα 17 — Στοιχειώδης Επεξεργασία Εικόνας

### Έξι βασικές πράξεις σε μία ματιά

Σε αυτή την ενότητα παρουσιάζονται έξι **θεμελιώδεις** μεταμορφώσεις που θα συναντάτε **παντού** στην ανάλυση εικόνας:

| Operation | Τι κάνει | Τυπική χρήση |
|-----------|----------|---------------|
| **Original** | Αναφορά (καμία αλλαγή) | Σύγκριση |
| **Histogram** | Κατανομή τιμών | Ποσοτική σύνοψη |
| **Normalization** | Κλιμάκωση σε [0, 1] | Pre-processing για ML |
| **Inversion** | `max − pixel` | Αλλαγή «πολικότητας» (π.χ. αρνητικό X-ray) |
| **Thresholding** | Δυαδικό: pixel > T → 1, αλλιώς 0 | Πρώτο βήμα segmentation |
| **Edge detection** | Ανίχνευση κλίσης | Ορισμός ορίων |

### Παιδαγωγικά σημεία ανά πράξη

#### Normalization
```python
img_norm = (img - img.min()) / (img.max() - img.min())
```
Λεγόμενη **min-max normalization**. Εξαιρετικά συνηθισμένη στο preprocessing για deep learning. **Προσοχή:** ευαίσθητη σε outliers — αν ένα pixel είναι ακραίο, «συμπιέζει» όλα τα άλλα. Εναλλακτική: **z-score** (`(img - mean) / std`) ή **percentile clipping** (κλιπ στο 1ο και 99ο percentile πριν τη normalization).

#### Inversion
```python
img_inv = img.max() - img
```
Σε **CT/X-ray**, οι ακτινολόγοι μερικές φορές προτιμούν inverted προβολή («negative») για να βλέπουν καλύτερα μαλακούς ιστούς.

#### Thresholding
```python
threshold = img.mean()
binary = img > threshold
```
Είναι η **απλούστερη μορφή segmentation**. Χρήσιμη όταν η εικόνα έχει σαφή διαχωρισμό ιστών (π.χ. οστά σε CT). Πιο sophisticated εκδοχές:
- **Otsu**: αυτόματη επιλογή κατωφλίου από το ιστόγραμμα.
- **Adaptive thresholding**: τοπικό κατώφλι που προσαρμόζεται σε διαφορετικές περιοχές.

#### Edge detection με gradient
```python
edges = |∂I/∂x| + |∂I/∂y|
```
Πολύ απλή προσέγγιση. Πιο εξελιγμένα:
- **Sobel filter**: σταθμισμένη πρώτη παράγωγος.
- **Canny edge detector**: full pipeline με smoothing, non-max suppression, double threshold.
- **Laplacian of Gaussian (LoG)**: ανίχνευση blob-like δομών.

> 💡 **Σύνδεση με προχωρημένες έννοιες:** Όλα τα παραπάνω είναι «κλασικές» μέθοδοι (pre-deep-learning). Σήμερα συχνά αντικαθίστανται από CNN-based μοντέλα (U-Net, nnU-Net) που μαθαίνουν την κατάλληλη μετατροπή από δεδομένα. Αλλά **οι βασικές αρχές παραμένουν** — και το να καταλαβαίνετε τις κλασικές μεθόδους κάνει καλύτερο deep learning practitioner.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              BASIC IMAGE PROCESSING                            ║
╚════════════════════════════════════════════════════════════════╝
""")

def demonstrate_image_processing(dicom_obj):
    """
    Demonstrate basic image processing operations
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Original
    axes[0, 0].imshow(img, cmap='gray')
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    # Histogram
    axes[0, 1].hist(img.ravel(), bins=50, color='blue', alpha=0.7)
    axes[0, 1].set_title('Histogram')
    axes[0, 1].set_xlabel('Pixel Intensity')
    axes[0, 1].set_ylabel('Frequency')
    
    # Normalized
    img_norm = (img - img.min()) / (img.max() - img.min())
    axes[0, 2].imshow(img_norm, cmap='gray')
    axes[0, 2].set_title('Normalized [0, 1]')
    axes[0, 2].axis('off')
    
    # Inverted
    img_inv = img.max() - img
    axes[1, 0].imshow(img_inv, cmap='gray')
    axes[1, 0].set_title('Inverted')
    axes[1, 0].axis('off')
    
    # Thresholded
    threshold = img.mean()
    img_thresh = img > threshold
    axes[1, 1].imshow(img_thresh, cmap='gray')
    axes[1, 1].set_title(f'Thresholded (>{threshold:.0f})')
    axes[1, 1].axis('off')
    
    # Edge detection (simple gradient)
    img_edges = np.abs(np.gradient(img)[0]) + np.abs(np.gradient(img)[1])
    axes[1, 2].imshow(img_edges, cmap='hot')
    axes[1, 2].set_title('Edge Detection')
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

demonstrate_image_processing(dicom_data)


## Ενότητα 18 — Αποθήκευση Εικόνων

### Γιατί να αποθηκεύσουμε;

Αφού επεξεργαστούμε ή αναλύσουμε μια εικόνα, συχνά θέλουμε:

- **Να τη μοιραστούμε** με συναδέλφους που δεν έχουν Python.
- **Να την εντάξουμε σε αναφορά** (PDF, παρουσίαση).
- **Να χτίσουμε dataset** για machine learning.
- **Να την επιστρέψουμε** στο PACS του νοσοκομείου ως νέο DICOM.

### Τρεις μορφές, τρεις σκοποί

| Format | Πότε το χρησιμοποιούμε | Διατηρεί metadata; |
|--------|--------------------------|---------------------|
| **PNG** | Για αναφορές, παρουσιάσεις, web | ❌ Όχι |
| **NumPy (`.npy`)** | Για επόμενη επεξεργασία σε Python | ❌ Όχι |
| **DICOM (`.dcm`)** | Για επιστροφή σε PACS, μοιραστείτε με ιατρικό λογισμικό | ✅ Ναι |

### PNG για παρουσιάσεις

```python
img_normalized = ((pixel_array - pixel_array.min()) /
                  (pixel_array.max() - pixel_array.min()) * 255).astype(np.uint8)
Image.fromarray(img_normalized).save("output.png")
```

> ⚠️ **Σημαντικό:** Το PNG είναι 8-bit (ή 16-bit). Αν αποθηκεύσετε CT απευθείας, θα **χάσετε ακρίβεια**. Αυτός είναι ο λόγος που πρώτα κανονικοποιούμε στο [0, 255]. Αυτό είναι αποδεκτό για **οπτικοποίηση** αλλά **όχι** για ποσοτική ανάλυση.

### NumPy για επόμενα στάδια

```python
np.save("output.npy", pixel_array)
# ... αργότερα ...
loaded = np.load("output.npy")
```

Διατηρείται το `dtype` και η ακρίβεια. Είναι το **πιο γρήγορο** format για επαναφόρτωση σε Python.

### DICOM για κλινική επιστροφή

```python
dicom_obj_copy = dicom_obj.copy()
dicom_obj_copy.SeriesDescription = "Processed by Student"
dicom_obj_copy.save_as("output.dcm")
```

Διατηρούνται όλα τα tags. **Σημαντικό:** σε σοβαρή χρήση, πρέπει να αλλάξετε:
- **`SOPInstanceUID`** (νέο μοναδικό UID για το νέο αρχείο).
- **`SeriesInstanceUID`** (νέα σειρά).
- Κάποιο **derivation flag** για να δηλώσετε ότι το αρχείο είναι παράγωγο.

Αλλιώς το PACS μπορεί να μπερδέψει τα νέα αρχεία με τα αρχικά.

### Κλειδί ασφαλείας

> 🚨 **ΠΑΝΤΑ** δουλεύετε σε **αντίγραφο** (`.copy()`). Ποτέ μην τροποποιείτε το original DICOM στη μνήμη και μετά το αποθηκεύετε με το ίδιο όνομα. Διατήρηση των πρωτότυπων δεδομένων είναι θεμελιώδης κανόνας.

### Κλειδί ιδιωτικότητας

> 🔐 **ΠΑΝΤΑ** ανωνυμοποιήστε πριν μοιραστείτε. Δείτε την Ενότητα 21 για το πώς. Ένα μόνο PNG που γλιστρά με όνομα ασθενούς στο burnin (κειμενική επικάλυψη) μπορεί να είναι παραβίαση GDPR.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              SAVING DICOM AND OTHER FORMATS                    ║
╚════════════════════════════════════════════════════════════════╝
""")

def save_dicom_as_formats(dicom_obj, output_prefix="output"):
    """
    Save DICOM image in different formats
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to save
    output_prefix : str
        Prefix for output files
    """
    
    pixel_array = dicom_obj.pixel_array
    
    # Normalize to 0-255 for standard image formats
    img_normalized = ((pixel_array - pixel_array.min()) / 
                      (pixel_array.max() - pixel_array.min()) * 255).astype(np.uint8)
    
    # Save as PNG
    from PIL import Image
    img_pil = Image.fromarray(img_normalized)
    png_path = f"{output_prefix}.png"
    img_pil.save(png_path)
    print(f"✓ Saved as PNG: {png_path}")
    
    # Save as NumPy array
    npy_path = f"{output_prefix}.npy"
    np.save(npy_path, pixel_array)
    print(f"✓ Saved as NumPy: {npy_path}")
    
    # Save modified DICOM (example: add a tag)
    dicom_obj_copy = dicom_obj.copy()
    dicom_obj_copy.SeriesDescription = "Processed by Student"
    dcm_path = f"{output_prefix}.dcm"
    dicom_obj_copy.save_as(dcm_path)
    print(f"✓ Saved as DICOM: {dcm_path}")

# Uncomment to save files
# save_dicom_as_formats(dicom_data, "my_processed_image")
print("(Saving functions defined - uncomment to use)")


## Ενότητα 19 — Αναζήτηση και Φιλτράρισμα Tags

### Το πρόβλημα: τα tags είναι ΠΟΛΛΑ

Ένα τυπικό DICOM περιέχει **50-300+ tags**. Πώς βρίσκετε αυτό που ψάχνετε;

### Η συνάρτηση `search_dicom_tags`

Δέχεται ένα DICOM και έναν **όρο αναζήτησης** (string), και επιστρέφει DataFrame με όλα τα tags των οποίων το όνομα **περιέχει** τον όρο:

```python
patient_tags = search_dicom_tags(dicom_data, 'Patient')
# → όλα τα tags με "Patient" στο όνομα: PatientName, PatientID, PatientBirthDate, ...
```

### Πώς λειτουργεί η iteration σε DICOM

Το `for elem in dicom_obj` διασχίζει **κάθε στοιχείο** του dataset. Κάθε `elem` έχει:

- `.name` — ανθρώπινα διαβάσιμο όνομα (π.χ. "Patient's Name")
- `.tag` — το (group, element) σε hex
- `.VR` — ο τύπος δεδομένων
- `.value` — η ίδια η τιμή

### Πρακτικές χρήσεις

| Όρος | Τι βρίσκετε |
|------|--------------|
| `'Patient'` | Δημογραφικά |
| `'Image'` | Παράμετροι εικόνας |
| `'Pixel'` | Pixel-σχετικά (`PixelSpacing`, `PixelData`, `BitsStored`...) |
| `'Date'` | Όλες οι ημερομηνίες (study, series, acquisition...) |
| `'Time'` | Ώρες |
| `'UID'` | Όλα τα μοναδικά αναγνωριστικά |
| `'Window'` | Default window settings |

### Παρατήρηση: τα `display(...)` calls

Στο Jupyter notebook, η `display()` είναι σαν `print()` αλλά «έξυπνη» για αντικείμενα όπως DataFrame — τα δείχνει σε **όμορφο HTML πίνακα** αντί για ASCII. Αν είστε σε plain Python script, χρησιμοποιήστε `print(df)` ή `df.to_string()`.

> 💡 **Tip:** Αν θέλετε να δείτε **όλα** τα tags ενός DICOM χωρίς φίλτρο, απλά κάντε `print(dicom_data)`. Παίρνετε ολόκληρη τη header.

> 🎯 **Άσκηση μνήμης:** Όταν συναντάτε ένα νέο tag, ψάξτε το στο **innolitics DICOM browser** (https://dicom.innolitics.com) — εκεί θα δείτε τη μορφή, τα όρια, το VR, και πού ανήκει στο πρότυπο.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              SEARCHING AND FILTERING TAGS                      ║
╚════════════════════════════════════════════════════════════════╝
""")

def search_dicom_tags(dicom_obj, search_term):
    """
    Search for DICOM tags containing a specific term
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object
    search_term : str
        Term to search for in tag names
    
    Returns:
    --------
    pd.DataFrame : Matching tags
    """
    
    matches = []
    
    for elem in dicom_obj:
        tag_name = elem.name
        if search_term.lower() in tag_name.lower():
            matches.append({
                'Tag Name': tag_name,
                'Tag': str(elem.tag),
                'VR': elem.VR,
                'Value': str(elem.value)[:50]  # Truncate long values
            })
    
    return pd.DataFrame(matches)

# Search for patient-related tags
print("\n🔍 Searching for 'Patient' tags:")
patient_tags = search_dicom_tags(dicom_data, 'Patient')
display(patient_tags)

# Search for image-related tags
print("\n🔍 Searching for 'Image' tags:")
image_tags = search_dicom_tags(dicom_data, 'Image')
display(image_tags)


## Ενότητα 20 — Ασκήσεις Πρακτικής (Μέρος Α)

### Πέντε ασκήσεις πάνω στις Ενότητες 1-19

Έχετε δει αρκετά. Τώρα είναι ώρα να **δουλέψετε μόνοι σας**. Αυτές οι ασκήσεις δεν λύνονται με αντιγραφή — απαιτούν να συνδέσετε **διαφορετικά κομμάτια** του υλικού.

### Σύντομες οδηγίες

| Άσκηση | Τι εξασκεί | Σχετικές ενότητες |
|--------|------------|-------------------|
| **1. Tag Exploration** | Ανάγνωση + φυσικές μετρήσεις | 4, 6, 16 |
| **2. Image Analysis** | Στατιστική + ιστόγραμμα + threshold | 9, 17 |
| **3. Series Navigation** | Φόρτωση φακέλου + montage + όγκος κάλυψης | 13, 14, 16 |
| **4. Window/Level** | Hounsfield + windows | 11, 12 |
| **5. Metadata Comparison** | Σύγκριση modalities | 5, 6, 8 |

### Πώς να δουλέψετε

> 1️⃣ **Διαβάστε όλη την άσκηση πριν αρχίσετε.** Εντοπίστε τι ακριβώς ζητείται.
> 2️⃣ **Σχεδιάστε το pipeline στα ελληνικά** σε χαρτί ή σε σχόλια. Ποια συνάρτηση έρχεται πρώτη;
> 3️⃣ **Γράψτε τον κώδικα βήμα-βήμα** και τρέξτε συχνά για να βλέπετε ενδιάμεσα αποτελέσματα.
> 4️⃣ **Επικυρώστε** με αναμενόμενες τιμές. Π.χ. αν μετράτε pixel σε mm, η τιμή πρέπει να είναι **λογική** (όγκος εγκεφάλου ≈ 1.4 L).
> 5️⃣ **Εξηγήστε γραπτώς** τι είδατε. Ο κώδικας **ΧΩΡΙΣ ερμηνεία** δεν είναι παράδοση — είναι μισή δουλειά.

### Παγίδες που να αποφύγετε

> ⚠️ **Παγίδα 1:** Να ξεχάσετε `pixel_array.astype(float)` πριν τις πράξεις. Σε `uint16`, ένα `pixel - 100` μπορεί να σας δώσει κενό λόγω overflow.

> ⚠️ **Παγίδα 2:** Να χρησιμοποιήσετε **raw pixel values** σε CT αντί για HU. Όλα σας τα νούμερα θα είναι λάθος.

> ⚠️ **Παγίδα 3:** Να μην ταξινομήσετε τις τομές πριν φτιάξετε τον 3D όγκο. Θα έχετε «παγωτό» από ασύνδετες τομές.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                    PRACTICE EXERCISES                          ║
╚════════════════════════════════════════════════════════════════╝

📝 EXERCISE 1: Tag Exploration
   - Load a DICOM file
   - Extract and print: Patient Name, Study Date, Modality
   - Calculate the physical size of the image in mm

📝 EXERCISE 2: Image Analysis
   - Load a DICOM image
   - Calculate: mean, median, standard deviation of pixel values
   - Create a histogram of pixel intensities
   - Apply a threshold to segment the image

📝 EXERCISE 3: Series Navigation
   - Load a series of DICOM files
   - Sort them by slice location
   - Create a montage view
   - Calculate the total volume coverage (in mm)

📝 EXERCISE 4: Window/Level
   - Load a CT image
   - Apply different window settings
   - Compare: soft tissue, lung, and bone windows
   - Explain what each window emphasizes

📝 EXERCISE 5: Metadata Comparison
   - Load multiple DICOM files from different modalities
   - Create a comparison table of key tags
   - Identify which tags are modality-specific
""")


## Ενότητα 21 — Βιβλιοθήκη Βοηθητικών Συναρτήσεων (κλάση `DICOMUtilities`)

### Γιατί τα ομαδοποιούμε σε κλάση;

Έχουμε τέσσερις διαφορετικές βοηθητικές λειτουργίες που **δεν χρειάζονται κατάσταση** (state). Είναι «καθαρά εργαλεία». Τα μαζεύουμε σε μια κλάση **για οργάνωση**, χρησιμοποιώντας `@staticmethod`:

- **`@staticmethod`** σημαίνει «αυτή η συνάρτηση δεν χρειάζεται `self`». Είναι σαν συνηθισμένη συνάρτηση, απλά «ζει» μέσα στην κλάση για namespace.

Έτσι μπορείτε να κάνετε `DICOMUtilities.anonymize_dicom(dcm)` ή `DICOMUtilities.calculate_slice_spacing(series)` χωρίς να χρειάζεται να φτιάξετε αντικείμενο.

### Οι τέσσερις λειτουργίες

#### 1. `anonymize_dicom` — αφαίρεση προσωπικών στοιχείων
**Νομικά κρίσιμη.** Αντικαθιστά PHI tags με 'ANONYMIZED':
- PatientName, PatientID, PatientBirthDate, PatientSex, PatientAge, PatientAddress

> 🚨 **ΠΡΟΣΟΧΗ — η συγκεκριμένη υλοποίηση είναι ΕΛΑΧΙΣΤΗ.** Πραγματική ανωνυμοποίηση πρέπει να καλύπτει:
> - **All PHI tags** (`InstitutionName`, `ReferringPhysicianName`, `OperatorsName`, `StudyID`, ...).
> - **Burnt-in pixel data** (κείμενο μέσα στην ίδια την εικόνα — ονόματα, ημερομηνίες). Σε ορισμένες ακολουθίες, εμφανίζονται **πάνω** στην εικόνα.
> - **Private tags** που πιθανόν περιέχουν proprietary metadata.
> - **UIDs** (πρέπει να αντικατασταθούν για να σπάσει η σύνδεση με την αρχική εξέταση).
>
> Για παραγωγική χρήση, δείτε εργαλεία όπως **dicognito**, **CTP** (DICOM Anonymizer), **DicomAnonymizerTool** — που εφαρμόζουν τις προδιαγραφές του DICOM standard για de-identification (PS3.15, Annex E).

#### 2. `get_image_orientation` — αξιακή/στεφανιαία/οβελιαία
Διαβάζει το `ImageOrientationPatient` (6 αριθμοί: row vector + column vector), υπολογίζει το **κάθετο διάνυσμα** (cross product), και επιστρέφει τον προσανατολισμό:

| Dominant axis | Προβολή |
|---------------|---------|
| X | Sagittal |
| Y | Coronal |
| Z | Axial |

> 📌 Αυτή είναι **απλοποίηση**. Σε λοξές (oblique) ακολουθίες, η εικόνα δεν είναι τέλεια ευθυγραμμισμένη με τους ανατομικούς άξονες — απλά παίρνουμε τον «κυρίαρχο» άξονα. Για ακριβή υπολογισμό χωρικού μετασχηματισμού, χρησιμοποιήστε SimpleITK.

#### 3. `calculate_slice_spacing` — απόσταση μεταξύ τομών
Υπολογίζει mean/std/min/max του spacing ταξινομώντας τις τομές κατά **z-coordinate**.

Γιατί std/min/max; Για να εντοπίσουμε **uneven spacing** — δηλαδή σειρές με «κενά» τομές που λείπουν.

#### 4. `quick_view` — γρήγορη οπτικοποίηση
Για να μην ξαναγράφετε `plt.imshow(...)` δέκα φορές την ώρα.

> 💡 **Παιδαγωγικό σημείο:** Ένα **utility module** σαν αυτό είναι από τα πρώτα πράγματα που χτίζει ένας έμπειρος προγραμματιστής όταν αρχίζει project. Δικές σας παραλλαγές που μπορείτε να γράψετε:
> - `dicom_diff(dcm1, dcm2)` — εμφανίζει διαφορές tags.
> - `validate_dicom(dcm)` — ελέγχει αν περιέχει όλα τα απαραίτητα tags για ML.
> - `bulk_summary(folder)` — γρήγορη σύνοψη όλων των DICOMs σε φάκελο.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                 UTILITY FUNCTIONS LIBRARY                      ║
╚════════════════════════════════════════════════════════════════╝
""")

class DICOMUtilities:
    """
    Collection of utility functions for DICOM processing
    """
    
    @staticmethod
    def anonymize_dicom(dicom_obj):
        """Remove patient identifiable information"""
        tags_to_anonymize = [
            'PatientName', 'PatientID', 'PatientBirthDate',
            'PatientSex', 'PatientAge', 'PatientAddress'
        ]
        
        for tag in tags_to_anonymize:
            if hasattr(dicom_obj, tag):
                setattr(dicom_obj, tag, 'ANONYMIZED')
        
        print("✓ Patient information anonymized")
        return dicom_obj
    
    @staticmethod
    def get_image_orientation(dicom_obj):
        """Determine image orientation (axial, coronal, sagittal)"""
        if not hasattr(dicom_obj, 'ImageOrientationPatient'):
            return "Unknown"
        
        iop = dicom_obj.ImageOrientationPatient
        
        # This is a simplified determination
        row_vec = np.array(iop[:3])
        col_vec = np.array(iop[3:])
        normal_vec = np.cross(row_vec, col_vec)
        
        # Find dominant direction
        abs_normal = np.abs(normal_vec)
        dominant = np.argmax(abs_normal)
        
        orientations = ['Sagittal', 'Coronal', 'Axial']
        return orientations[dominant]
    
    @staticmethod
    def calculate_slice_spacing(dicom_series):
        """Calculate spacing between slices"""
        if len(dicom_series) < 2:
            return None
        
        positions = []
        for dcm in dicom_series:
            if hasattr(dcm, 'ImagePositionPatient'):
                positions.append(dcm.ImagePositionPatient[2])  # Z coordinate
        
        if len(positions) < 2:
            return None
        
        positions.sort()
        spacings = np.diff(positions)
        
        return {
            'mean_spacing': np.mean(spacings),
            'std_spacing': np.std(spacings),
            'min_spacing': np.min(spacings),
            'max_spacing': np.max(spacings)
        }
    
    @staticmethod
    def quick_view(dicom_obj, title=""):
        """Quick visualization of DICOM image"""
        plt.figure(figsize=(8, 8))
        plt.imshow(dicom_obj.pixel_array, cmap='gray')
        plt.title(f"{title}\n{dicom_obj.get('Modality', 'Unknown')} "
                  f"{dicom_obj.Rows}x{dicom_obj.Columns}")
        plt.colorbar()
        plt.tight_layout()
        plt.show()

# Demonstrate utilities
utils = DICOMUtilities()

print("\n🔧 Demonstrating utility functions:")
print(f"Image Orientation: {utils.get_image_orientation(dicom_data)}")

utils.quick_view(dicom_data, "Quick View Demo")


## Ενότητα 22 — Συνηθισμένες Παγίδες & Καλές Πρακτικές

### Defensive programming — η νοοτροπία

Ο κώδικας ιατρικής εικόνας **πρέπει** να είναι ανθεκτικός. Σε κλινικά δεδομένα, θα συναντήσετε:
- Tags που λείπουν.
- Λανθασμένες ή κενές τιμές.
- Παλιά αρχεία με «παράξενα» VRs.
- Ιδιαιτερότητες κατασκευαστών (Siemens vs GE vs Philips).

Ο κώδικας πρέπει να **προβλέπει** όλα αυτά, όχι να κρασάρει στην πρώτη παρατυπία.

### Οι πέντε κορυφαίες παγίδες

#### Παγίδα 1: Πρόσβαση χωρίς έλεγχο
```python
# ❌ ΚΑΚΟ
name = dicom_obj.PatientName  # Κρασάρει αν λείπει

# ✅ ΚΑΛΟ
name = dicom_obj.get('PatientName', 'Unknown')
```

#### Παγίδα 2: Pixel αντί για mm
```python
# ❌ ΚΑΚΟ
print(f"Tumor diameter: {tumor_pixels} pixels")

# ✅ ΚΑΛΟ
spacing_mm = dicom_obj.PixelSpacing[0]
print(f"Tumor diameter: {tumor_pixels * spacing_mm:.1f} mm")
```

#### Παγίδα 3: CT χωρίς HU conversion
```python
# ❌ ΚΑΚΟ — αυτό το νούμερο δεν έχει νόημα
mean_intensity = dicom_obj.pixel_array.mean()

# ✅ ΚΑΛΟ
hu = dicom_obj.pixel_array * dicom_obj.RescaleSlope + dicom_obj.RescaleIntercept
mean_hu = hu.mean()  # τώρα έχει φυσικό νόημα
```

#### Παγίδα 4: Υπόθεση ότι όλα τα tags υπάρχουν
```python
# ❌ ΚΑΚΟ
thickness = dicom_obj.SliceThickness

# ✅ ΚΑΛΟ
if hasattr(dicom_obj, 'SliceThickness'):
    thickness = dicom_obj.SliceThickness
else:
    thickness = None  # ή υπολογίστε από ImagePositionPatient
```

#### Παγίδα 5: Μη ταξινομημένη σειρά
```python
# ❌ ΚΑΚΟ — τυχαία σειρά
volume = np.stack([pydicom.dcmread(f).pixel_array for f in os.listdir(folder)])

# ✅ ΚΑΛΟ
files = [pydicom.dcmread(f) for f in folder]
files.sort(key=lambda d: float(d.ImagePositionPatient[2]))
volume = np.stack([f.pixel_array for f in files])
```

### Επτά καλές πρακτικές

1. **Επικυρώνετε τα DICOM πριν την επεξεργασία** — έλεγχος `Modality`, διαστάσεων, απαραίτητων tags.
2. **Χρησιμοποιείτε try-except** για robustness — ειδικά σε batch processing.
3. **Παρακολουθείτε coordinate systems** — pixel coords vs patient coords vs world coords.
4. **Καταγράφετε τα window/level settings** στις παρουσιάσεις.
5. **Ποτέ μην τροποποιείτε τα original** — δουλέψτε σε copies.
6. **Ανωνυμοποιείτε πριν τη διανομή** — και θεωρήστε ότι κάθε file leak είναι πρόβλημα.
7. **Ελέγχετε modality** πριν εφαρμόσετε modality-specific operations (HU σε CT, T1/T2 σε MRI...).

> 🎯 **Η μάντρα του developer ιατρικής εικόνας:** «Αν μπορεί να πάει στραβά, **θα** πάει στραβά σε κάποιον ασθενή κάποια στιγμή. Γράψε τον κώδικα προβλέποντάς το.»


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              COMMON PITFALLS AND TIPS                          ║
╚════════════════════════════════════════════════════════════════╝

⚠️ COMMON PITFALLS:

1. Not checking if tags exist before accessing
   ❌ value = dicom_obj.PatientName  # May crash
   ✅ value = dicom_obj.get('PatientName', 'N/A')

2. Ignoring pixel spacing for measurements
   ❌ distance = 100 pixels
   ✅ distance = 100 pixels * pixel_spacing mm

3. Not converting to Hounsfield Units for CT
   ❌ Using raw pixel values for CT analysis
   ✅ Apply RescaleSlope and RescaleIntercept

4. Assuming all tags are present
   ❌ Using attributes directly
   ✅ Check with hasattr() or use get()

5. Not sorting slices in a series
   ❌ Processing slices in random order
   ✅ Sort by InstanceNumber or ImagePositionPatient

💡 BEST PRACTICES:

1. Always validate DICOM files before processing
2. Use try-except blocks for robust code
3. Keep track of coordinate systems
4. Document your window/level settings
5. Preserve original data - work on copies
6. Anonymize data when sharing
7. Check modality before applying modality-specific processing
""")


## Ενότητα 23 — Πρόσθετοι Πόροι

### Επίσημη τεκμηρίωση

- **PyDICOM**: https://pydicom.github.io/ — αναφορά API, παραδείγματα, FAQ.
- **DICOM Standard**: https://www.dicomstandard.org/ — η «βίβλος». Μεγάλη, αλλά πλήρης.
- **DICOM Innolitics Browser**: https://dicom.innolitics.com — ο πιο φιλικός τρόπος εξερεύνησης του προτύπου.

### Δημόσια datasets

- **The Cancer Imaging Archive (TCIA)** — https://www.cancerimagingarchive.net/
- **Medical Segmentation Decathlon** — http://medicaldecathlon.com/
- **Grand Challenge** — https://grand-challenge.org/ (datasets από διαγωνισμούς)
- **ADNI** (Alzheimer's) — https://adni.loni.usc.edu/
- **OASIS** (νευροαπεικόνιση) — https://www.oasis-brains.org/
- **MIMIC-CXR** (chest X-rays + reports) — https://physionet.org/

### Βιβλιοθήκες πέρα από PyDICOM

| Βιβλιοθήκη | Σε τι διαφέρει | Πότε προτιμάται |
|------------|------------------|------------------|
| **SimpleITK** | Πιο ώριμη για 3D processing, registration, segmentation | Παραγωγικά pipelines |
| **nibabel** | Διαβάζει NIfTI (κοινό format σε νευροαπεικόνιση) | fMRI, BIDS datasets |
| **scikit-image** | Γενική εικόνα-processing | Filtering, segmentation, features |
| **MONAI** | Deep learning για ιατρικές εικόνες (PyTorch-based) | Modern ML projects |
| **TorchIO** | Augmentations για ιατρικά volumes | Training data pipelines |
| **3D Slicer** | Λογισμικό-εργαστήριο με Python API | Διαδραστική ανάλυση |

### Μάθηση & βίντεο

- **PyDICOM tutorials** στο GitHub.
- **MONAI bootcamp materials** (YouTube).
- **Coursera: AI for Medicine Specialization** (deeplearning.ai).
- **edX: Medical Image Analysis** (διαφορετικά πανεπιστήμια).

### Κλινικά πρότυπα

- **DICOM PS3.15** (Annex E): de-identification.
- **DICOM PS3.10**: file format.
- **IHE Profiles**: συμβατότητα ροών εργασίας.

> 💡 **Συμβουλή για συνεχιζόμενη ενημέρωση:** Ακολουθήστε στα social media: **@PyDICOM**, **@MONAI_io**, και τους ερευνητές που δουλεύουν στο πεδίο σας. Το πεδίο εξελίσσεται γρήγορα, ειδικά με την έλευση των foundation models στην ιατρική εικόνα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  ADDITIONAL RESOURCES                          ║
╚════════════════════════════════════════════════════════════════╝

📚 DOCUMENTATION:
   - PyDICOM: https://pydicom.github.io/
   - DICOM Standard: https://www.dicomstandard.org/

📊 PUBLIC DATASETS:
   - The Cancer Imaging Archive (TCIA): https://www.cancerimagingarchive.net/
   - Medical Segmentation Decathlon: http://medicaldecathlon.com/

🛠️ USEFUL LIBRARIES:
   - SimpleITK: Advanced medical image processing
   - nibabel: Working with neuroimaging formats
   - scikit-image: General image processing
   - 3D Slicer: Visualization and analysis platform

📖 LEARNING:
   - DICOM Tutorial: https://www.dicomstandard.org/
   - Medical Imaging courses on Coursera, edX
   - PyDICOM documentation and examples
""")


## Ενότητα 24 — Σύνοψη Μέρους Α & Επόμενα Βήματα

### Τι μάθατε σε αυτό το πρώτο μέρος

Στις Ενότητες 1-23 καλύψαμε όλα τα **απαραίτητα** για να ξεκινήσετε να εργάζεστε με DICOM:

**Έννοιες**
- ✓ Τι είναι το DICOM και γιατί είναι σημαντικό
- ✓ Η ιεραρχία Patient → Study → Series → Image
- ✓ Τι είναι τα tags, VRs, modules
- ✓ Hounsfield Units και η φυσική σημασία τους
- ✓ Pixel spacing και πραγματικές μετρήσεις

**Δεξιότητες**
- ✓ Φόρτωση και ανάγνωση DICOM με `pydicom.dcmread`
- ✓ Πρόσβαση σε tags με τέσσερις διαφορετικούς τρόπους
- ✓ Οπτικοποίηση εικόνων με `matplotlib`
- ✓ Windowing για βελτίωση αντίθεσης
- ✓ Φόρτωση και πλοήγηση σειρών
- ✓ Δημιουργία 3D όγκων και ανατομικών προβολών (axial/coronal/sagittal)
- ✓ Στοιχειώδης image processing
- ✓ Αποθήκευση σε διάφορες μορφές
- ✓ Defensive programming για κλινικά δεδομένα

### Τι έρχεται στο Μέρος Β (Ενότητες 26-36)

Στο επόμενο μέρος προχωράμε σε **βάθος ανάλυσης**:

- **Στατιστική ανάλυση** ιστογραμμάτων (mean, median, skewness, kurtosis)
- **Συγκριτική οπτικοποίηση** (box plots, violin plots, Q-Q plots)
- **Χωρική ανάλυση** ανά περιοχή
- **Προφίλ έντασης** και ανίχνευση κλίσεων
- **Histogram equalization** για βελτίωση αντίθεσης
- **Ποσοτικοποίηση αντίθεσης/φωτεινότητας**
- **Εκτίμηση και αξιολόγηση θορύβου** (SNR)
- **Σύνταξη ολοκληρωμένης αναφοράς ποιότητας**

### Πώς να συνεχίσετε εκτός μαθήματος

> 🎯 **1ο βήμα — Εξάσκηση με πραγματικά δεδομένα.** Κατεβάστε ένα μικρό dataset από TCIA (π.χ. **LIDC-IDRI** για lung CT). Εφαρμόστε όλο το pipeline που μάθατε.

> 🎯 **2ο βήμα — Επιλέξτε modality εξειδίκευσης.** CT για ογκολογία, MRI για νευρολογία/μυοσκελετικό, X-ray για cardio-pulmonary, US για μαιευτική. Κάθε ένα έχει τις δικές του ιδιαιτερότητες.

> 🎯 **3ο βήμα — Πραγματικό project.** Διαλέξτε μια ερευνητική ή κλινική ερώτηση. Γράψτε ολόκληρο pipeline από φόρτωση μέχρι αναφορά. **Αυτό** είναι που μένει στη μνήμη — όχι τα notebooks.

> 🎯 **4ο βήμα — Συμβολή σε open source.** Το PyDICOM, το MONAI, το SimpleITK δέχονται contributions. Είναι ο καλύτερος τρόπος να βελτιωθείτε ως developer ιατρικής εικόνας.

### Πρόκληση τέλους

Φτιάξτε **τον δικό σας ολοκληρωμένο DICOM viewer** που να συνδυάζει τα παρακάτω:

- Πλοήγηση πολλαπλών τομών (slider)
- Διαδραστική ρύθμιση window/level
- Εργαλεία μέτρησης (ευθεία γραμμή σε mm)
- Απλή annotation
- Εξαγωγή σε PNG/PDF report

Θα μάθετε **πραγματικά** πολλά από αυτή την άσκηση.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  SUMMARY & NEXT STEPS                          ║
╚════════════════════════════════════════════════════════════════╝

🎓 WHAT YOU'VE LEARNED:

✓ What DICOM files are and why they're important
✓ How to load and read DICOM files with PyDICOM
✓ Understanding and accessing DICOM tags
✓ Visualizing medical images
✓ Working with Hounsfield Units (CT)
✓ Applying window/level adjustments
✓ Loading and navigating image series
✓ Creating 3D volumes from slices
✓ Basic image processing operations
✓ Saving and exporting DICOM data

🚀 NEXT STEPS:

1. Practice with different modalities (CT, MRI, X-ray)
2. Explore advanced segmentation techniques
3. Learn 3D visualization methods
4. Study image registration and fusion
5. Implement machine learning pipelines
6. Work with real clinical datasets
7. Build your own analysis tools

💪 CHALLENGE YOURSELF:
   Try to build a complete DICOM viewer with:
   - Multi-slice navigation
   - Interactive window/level adjustment
   - Measurement tools
   - Annotation capabilities
   - Export functionality

Good luck with your medical image analysis journey! 🏥
""")


## Ενότητα 25 — Τελική Επίδειξη: Dashboard Σύνοψης

### Συνθέτοντας όλα όσα μάθαμε

Σε αυτή την τελευταία ενότητα του Μέρους Α, η συνάρτηση `create_comprehensive_summary` δημιουργεί ένα **dashboard** που συνδυάζει σχεδόν **όλες** τις τεχνικές των προηγούμενων ενοτήτων σε ένα σχήμα.

### Τι περιέχει το dashboard

| Στοιχείο | Σχετική ενότητα | Τι μας λέει |
|----------|------------------|--------------|
| **Κύρια εικόνα** | 10 | Η ίδια η DICOM τομή με titlebar |
| **Ιστόγραμμα έντασης** | 17 | Κατανομή τιμών pixel |
| **Πίνακας στατιστικών** | 9, 17 | Mean, median, std, min, max |
| **Πίνακας metadata** | 5, 6, 8 | Modality, ημερομηνία |
| **Προφίλ έντασης (μεσαία γραμμή)** | (πρόδρομος του 30) | Πώς αλλάζει η ένταση |
| **Downsampled view** | 17 | Compressed visualization |

### Παρατηρήστε τη χρήση `gridspec`

```python
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
```

Αντί για `subplots(3, 3)` που δίνει **ισομεγέθη** κουτάκια, το `gridspec` σας επιτρέπει **διαφορετικά μεγέθη** για κάθε panel. Στο παράδειγμα:

- `gs[0:2, 0:2]` → η κύρια εικόνα καταλαμβάνει 2×2 = 4 cells.
- `gs[0, 2]` → το ιστόγραμμα μόνο 1 cell.
- `gs[2, :]` → το προφίλ καταλαμβάνει όλη την κάτω γραμμή.

> 💡 **Όταν η σχεδίαση μετρά:** Οι ιατρικές αναφορές πρέπει να είναι **εύληπτες με μια ματιά**. Το gridspec είναι το εργαλείο για επαγγελματικά layouts.

### Από εδώ προχωράμε στο Μέρος Β

Τώρα ξέρετε **πώς να φτάσετε στα pixel και στα metadata**. Στις Ενότητες 26-36 θα μάθετε **πώς να τα αναλύσετε σε βάθος** — με στατιστικές, ιστογράμματα, ποιοτικές μετρικές, και αναφορές παραγωγικού επιπέδου.

> 🎓 **Στο Μέρος Β θα συνεχίσουμε να χρησιμοποιούμε τη μεταβλητή `dicom_data`** που φορτώθηκε στην Ενότητα 4. Αν τρέξατε όλα τα κελιά μέχρι εδώ, είστε έτοιμοι. Αν επανεκκινήσετε τον kernel, θα χρειαστεί να ξανατρέξετε τουλάχιστον τις Ενότητες 1, 3 και 4.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              INTERACTIVE DEMO COMPLETE!                        ║
╚════════════════════════════════════════════════════════════════╝

You now have a complete toolkit for working with DICOM images!

Try modifying the code above to:
- Load your own DICOM files
- Create custom visualizations
- Implement new analysis functions
- Build your own viewer

Remember: Practice makes perfect! 💻
""")

# Create a final comprehensive visualization
def create_comprehensive_summary(dicom_obj):
    """
    Create a comprehensive visualization summarizing all learned concepts
    """
    
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # Main image
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    ax1.imshow(dicom_obj.pixel_array, cmap='gray')
    ax1.set_title(f'DICOM Image: {dicom_obj.Modality}\n'
                  f'{dicom_obj.Rows}x{dicom_obj.Columns} pixels', 
                  fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # Histogram
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(dicom_obj.pixel_array.ravel(), bins=50, color='steelblue', alpha=0.7)
    ax2.set_title('Intensity Histogram', fontsize=10)
    ax2.set_xlabel('Pixel Value')
    ax2.set_ylabel('Frequency')
    ax2.grid(True, alpha=0.3)
    
    # Statistics
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.axis('off')
    stats_text = f"""
    IMAGE STATISTICS
    ───────────────
    Mean: {dicom_obj.pixel_array.mean():.1f}
    Median: {np.median(dicom_obj.pixel_array):.1f}
    Std Dev: {dicom_obj.pixel_array.std():.1f}
    Min: {dicom_obj.pixel_array.min()}
    Max: {dicom_obj.pixel_array.max()}
    
    METADATA
    ───────────────
    Modality: {dicom_obj.get('Modality', 'N/A')}
    Date: {dicom_obj.get('StudyDate', 'N/A')}
    """
    ax3.text(0.1, 0.5, stats_text, fontsize=9, family='monospace',
             verticalalignment='center')
    
    # Profile plot
    ax4 = fig.add_subplot(gs[2, :2])
    mid_row = dicom_obj.Rows // 2
    profile = dicom_obj.pixel_array[mid_row, :]
    ax4.plot(profile, color='darkred', linewidth=2)
    ax4.set_title(f'Intensity Profile (Row {mid_row})', fontsize=10)
    ax4.set_xlabel('Column')
    ax4.set_ylabel('Intensity')
    ax4.grid(True, alpha=0.3)
    
    # 2D histogram
    ax5 = fig.add_subplot(gs[2, 2])
    # Create a simple 2D view of pixel correlations
    small_img = dicom_obj.pixel_array[::10, ::10]  # Downsample for speed
    ax5.imshow(small_img, cmap='viridis', aspect='auto')
    ax5.set_title('Downsampled View', fontsize=10)
    ax5.axis('off')
    
    plt.suptitle('DICOM Analysis Summary Dashboard', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.show()

# Create final summary
print("\n" + "="*70)
print("CREATING COMPREHENSIVE SUMMARY VISUALIZATION...")
print("="*70)
create_comprehensive_summary(dicom_data)

print("\n🎉 Tutorial Complete! You're now ready to work with DICOM images! 🎉")


## Ενότητα 26 — Κατανόηση Ιστογραμμάτων Εικόνας

### Τι είναι μια ιατρική εικόνα στον υπολογιστή;

Πριν μιλήσουμε για ιστογράμματα, ας ξεκαθαρίσουμε τι «βλέπει» πραγματικά ο υπολογιστής. Μια εικόνα DICOM (CT, MRI, mammography κ.λπ.) είναι **ένας πίνακας αριθμών**. Κάθε θέση του πίνακα αντιστοιχεί σε ένα **pixel** (εικονοστοιχείο) και ο αριθμός εκεί λέγεται **ένταση** (intensity). Στην ακτινολογία, αυτή η ένταση συνδέεται με φυσικά μεγέθη — π.χ. στο CT αντιστοιχεί σε **Hounsfield Units** (πυκνότητα ιστού), στο MRI εξαρτάται από το είδος της ακολουθίας (T1, T2, DWI κ.λπ.).

Άρα όταν λέμε «ανάλυση εικόνας», στην πράξη κάνουμε στατιστική σε εκατομμύρια αριθμούς.

### Τι είναι το ιστόγραμμα;

Το **ιστόγραμμα** είναι ένα διάγραμμα που δείχνει **πόσα pixel έχουν την κάθε τιμή έντασης**. Δηλαδή:

- Στον **οριζόντιο άξονα** (x): οι τιμές έντασης (από τη μικρότερη στη μεγαλύτερη).
- Στον **κατακόρυφο άξονα** (y): πόσες φορές εμφανίζεται η κάθε τιμή στην εικόνα.

> 📌 **Διαισθητική αναλογία:** Φανταστείτε ότι ρίχνετε όλα τα pixel σε «κουτάκια» (bins) ανάλογα με την τιμή τους — π.χ. ένα κουτί για τιμές 0–10, ένα για 10–20, κ.ο.κ. Το ιστόγραμμα είναι απλά το ύψος αυτών των κουτιών.

### Γιατί έχει σημασία στην ιατρική;

Διαφορετικοί ιστοί έχουν διαφορετικές εντάσεις. Σε ένα CT θώρακος, για παράδειγμα, ο **αέρας** δίνει πολύ χαμηλές τιμές, το **μαλακό μόριο** μεσαίες, και το **οστό** πολύ υψηλές. Όταν δούμε ένα ιστόγραμμα με δύο ή τρεις «κορυφές» (peaks/modes), αυτό μας λέει ότι υπάρχουν στην εικόνα διακριτοί ιστοί — μια **πολυτροπική** (multimodal) κατανομή.

### Τι θα δείτε στον κώδικα παρακάτω

Η συνάρτηση `comprehensive_histogram_analysis` θα παρουσιάσει **έξι διαφορετικές αναπαραστάσεις** του ίδιου ιστογράμματος. Καθεμιά απαντά σε διαφορετική ερώτηση:

1. **Βασικό ιστόγραμμα** — Πώς κατανέμονται γενικά οι εντάσεις;
2. **Ιστόγραμμα με στατιστικές γραμμές** — Πού πέφτει η μέση τιμή, η διάμεσος, και το ±1 τυπική απόκλιση;
3. **Αθροιστικό ιστόγραμμα (CDF)** — Τι ποσοστό των pixel βρίσκεται κάτω από κάθε τιμή; (Βασικό εργαλείο για windowing/thresholding.)
4. **Ιστόγραμμα σε λογαριθμική κλίμακα** — Αν λίγα pixel έχουν ακραίες τιμές, η γραμμική κλίμακα τα «κρύβει». Ο λογάριθμος τα φέρνει στην επιφάνεια.
5. **Κανονικοποιημένο ιστόγραμμα (PDF)** — Αντί για πλήθος, βλέπουμε **πιθανότητα**. Έτσι ιστογράμματα από εικόνες διαφορετικού μεγέθους γίνονται συγκρίσιμα.
6. **Ιστόγραμμα με εκατοστημόρια** — Πού πέφτουν τα 1ο, 5ο, 25ο, 50ο, 75ο, 95ο, 99ο εκατοστημόριο. Αυτό μας βοηθά να εντοπίσουμε ακραίες τιμές (outliers) και να επιλέξουμε σωστό window/level για την προβολή.

> ⚠️ **Σημαντικό:** Ο αριθμός των bins (εδώ 50) επηρεάζει την εμφάνιση. Λίγα bins → πολύ «λεία» εικόνα, χάνουμε λεπτομέρεια. Πολλά bins → θορυβώδες ιστόγραμμα. Στις ασκήσεις θα δείτε μόνοι σας πώς αλλάζει η ερμηνεία ανάλογα με την επιλογή.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║            UNDERSTANDING IMAGE HISTOGRAMS                      ║
╚════════════════════════════════════════════════════════════════╝

A histogram shows the distribution of pixel intensities in an image.
- X-axis: Pixel intensity values
- Y-axis: Number of pixels with that intensity

Why histograms matter:
✓ Understanding image contrast
✓ Detecting image quality issues
✓ Choosing appropriate window/level settings
✓ Identifying different tissue types
✓ Preprocessing decisions (normalization, etc.)
""")

def comprehensive_histogram_analysis(dicom_obj, title=""):
    """
    Create comprehensive histogram visualizations
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to analyze
    title : str
        Title for the analysis
    """
    
    pixel_array = dicom_obj.pixel_array.flatten()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # 1. Basic Histogram
    axes[0, 0].hist(pixel_array, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 0].set_title('Basic Histogram', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Pixel Intensity')
    axes[0, 0].set_ylabel('Frequency')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Histogram with Statistics Lines
    axes[0, 1].hist(pixel_array, bins=50, color='lightcoral', alpha=0.7, edgecolor='black')
    mean_val = pixel_array.mean()
    median_val = np.median(pixel_array)
    std_val = pixel_array.std()
    
    axes[0, 1].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.1f}')
    axes[0, 1].axvline(median_val, color='green', linestyle='--', linewidth=2, label=f'Median: {median_val:.1f}')
    axes[0, 1].axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=2, label=f'+1 SD: {mean_val+std_val:.1f}')
    axes[0, 1].axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=2, label=f'-1 SD: {mean_val-std_val:.1f}')
    
    axes[0, 1].set_title('Histogram with Statistics', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Pixel Intensity')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].legend(fontsize=9)
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Cumulative Histogram
    counts, bins = np.histogram(pixel_array, bins=100)
    cumulative = np.cumsum(counts)
    axes[0, 2].plot(bins[:-1], cumulative, color='darkgreen', linewidth=2)
    axes[0, 2].fill_between(bins[:-1], cumulative, alpha=0.3, color='lightgreen')
    axes[0, 2].set_title('Cumulative Histogram', fontsize=12, fontweight='bold')
    axes[0, 2].set_xlabel('Pixel Intensity')
    axes[0, 2].set_ylabel('Cumulative Frequency')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Log Scale Histogram
    axes[1, 0].hist(pixel_array, bins=50, color='purple', alpha=0.7, edgecolor='black')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_title('Histogram (Log Scale)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Pixel Intensity')
    axes[1, 0].set_ylabel('Frequency (log scale)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Normalized Histogram (Probability Density)
    axes[1, 1].hist(pixel_array, bins=50, density=True, color='teal', alpha=0.7, edgecolor='black')
    axes[1, 1].set_title('Normalized Histogram (PDF)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Pixel Intensity')
    axes[1, 1].set_ylabel('Probability Density')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Percentile Analysis
    percentiles = [1, 5, 25, 50, 75, 95, 99]
    percentile_values = np.percentile(pixel_array, percentiles)
    
    axes[1, 2].hist(pixel_array, bins=50, color='goldenrod', alpha=0.5, edgecolor='black')
    for p, val in zip(percentiles, percentile_values):
        axes[1, 2].axvline(val, color='red', linestyle='--', linewidth=1, alpha=0.7)
        axes[1, 2].text(val, axes[1, 2].get_ylim()[1]*0.9, f'{p}%', 
                       rotation=90, fontsize=8, va='top')
    
    axes[1, 2].set_title('Histogram with Percentiles', fontsize=12, fontweight='bold')
    axes[1, 2].set_xlabel('Pixel Intensity')
    axes[1, 2].set_ylabel('Frequency')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.suptitle(f'Comprehensive Histogram Analysis: {title}', 
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

# Demonstrate histogram analysis
comprehensive_histogram_analysis(dicom_data, f"{dicom_data.Modality} Image")


## Ενότητα 27 — Αναλυτική Στατιστική Ανάλυση

### Γιατί χρειαζόμαστε αριθμούς-σύνοψη;

Μια εικόνα 512×512 έχει 262.144 pixel. Δεν μπορούμε να μιλάμε για κάθε ένα ξεχωριστά — χρειαζόμαστε **συνοπτικά μέτρα** που περιγράφουν την κατανομή σε λίγους αριθμούς. Αυτά τα μέτρα είναι το ABC της ανάλυσης: μια εικόνα μπορεί να συγκριθεί με μια άλλη, ένας ιστός με έναν άλλον, μια εξέταση με μια προηγούμενη.

### Οι τέσσερις οικογένειες μέτρων

#### 1. Μέτρα κεντρικής τάσης — *Πού «κάθεται» η κατανομή;*

- **Μέση τιμή (mean, μ):** Το άθροισμα διά το πλήθος. Ευαίσθητη σε ακραίες τιμές.
- **Διάμεσος (median):** Η μεσαία τιμή όταν τα pixel ταξινομηθούν. **Ανθεκτική** σε outliers — γι' αυτό προτιμάται όταν υπάρχει θόρυβος.
- **Επικρατούσα τιμή (mode):** Η πιο συχνή τιμή. Σε ιατρικές εικόνες συχνά αντιστοιχεί στον κυρίαρχο ιστό ή στο φόντο.

> 🧠 **Κανόνας του φοιτητή:** Αν η μέση τιμή και η διάμεσος διαφέρουν σημαντικά, η κατανομή σας **δεν είναι συμμετρική**.

#### 2. Μέτρα διασποράς — *Πόσο «απλωμένη» είναι η κατανομή;*

- **Τυπική απόκλιση (std, σ):** Πόσο μακριά, κατά μέσο όρο, βρίσκονται τα pixel από τη μέση τιμή. Σε ιατρικές εικόνες αντικατοπτρίζει ταυτόχρονα την **ετερογένεια ιστών** ΚΑΙ τον **θόρυβο**.
- **Διακύμανση (variance, σ²):** Το τετράγωνο της τυπικής απόκλισης.
- **Εύρος (range):** max − min. Πολύ ευαίσθητο — αρκεί ένα κακό pixel για να το «τινάξει στον αέρα».
- **Ενδοτεταρτημοριακό εύρος (IQR):** Q3 − Q1. **Ανθεκτικό** μέτρο διασποράς, ισοδύναμο της διαμέσου.

#### 3. Μέτρα σχήματος — *Πώς «μοιάζει» η κατανομή;*

- **Ασυμμετρία (skewness):** Πόσο γέρνει η κατανομή προς τη μία πλευρά.
  - skew ≈ 0 → συμμετρική (όπως κανονική κατανομή)
  - skew > 0 → δεξιά ασυμμετρία (μακριά ουρά προς τα δεξιά, λιγοστά λαμπρά pixel)
  - skew < 0 → αριστερή ασυμμετρία
- **Κύρτωση (kurtosis):** Πόσο «μυτερή» ή «βαριά στις ουρές» είναι.
  - kurtosis > 0 → πιο μυτερή κορυφή και βαρύτερες ουρές (περισσότερα ακραία pixel)
  - kurtosis < 0 → πιο πλατιά κατανομή

#### 4. Μέτρα ποιότητας

- **Συντελεστής μεταβλητότητας (CV = σ/μ × 100%):** Σχετική διασπορά. Επιτρέπει σύγκριση μεταξύ εικόνων με διαφορετικές κλίμακες.
- **SNR (Signal-to-Noise Ratio):** Στην απλή του μορφή, μ/σ. Όσο μεγαλύτερο, τόσο «καθαρότερη» η εικόνα.

### Τι κάνει ο κώδικας

Δύο συναρτήσεις:

1. **`comprehensive_statistical_analysis`** — Υπολογίζει όλα τα παραπάνω σε ένα dictionary.
2. **`display_statistics_report`** — Παρουσιάζει τα αποτελέσματα σε καλά μορφοποιημένο πίνακα και προσθέτει **αυτόματη ερμηνεία** (π.χ. «Right-skewed»).

> 📌 Παρατηρήστε ότι η συνάρτηση επιστρέφει dictionary. Αυτή είναι καλή πρακτική: ο κώδικας που υπολογίζει διαχωρίζεται από τον κώδικα που εμφανίζει — μπορείτε να ξαναχρησιμοποιήσετε τα νούμερα σε άλλη ανάλυση χωρίς να ξανατρέξετε τα ίδια.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              DETAILED STATISTICAL ANALYSIS                     ║
╚════════════════════════════════════════════════════════════════╝

Statistical measures help us understand:
✓ Central tendency (mean, median, mode)
✓ Spread/dispersion (standard deviation, variance, range)
✓ Distribution shape (skewness, kurtosis)
✓ Outliers and extreme values
✓ Image quality and consistency
""")

def comprehensive_statistical_analysis(dicom_obj):
    """
    Perform comprehensive statistical analysis on DICOM image
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to analyze
    
    Returns:
    --------
    dict : Dictionary containing all statistical measures
    """
    
    pixel_array = dicom_obj.pixel_array.flatten()
    
    # Basic statistics
    stats = {
        'count': len(pixel_array),
        'mean': np.mean(pixel_array),
        'median': np.median(pixel_array),
        'mode': float(np.bincount(pixel_array.astype(int)).argmax()) if pixel_array.max() < 10000 else 'N/A',
        'std': np.std(pixel_array),
        'variance': np.var(pixel_array),
        'min': np.min(pixel_array),
        'max': np.max(pixel_array),
        'range': np.ptp(pixel_array),  # peak to peak (max - min)
    }
    
    # Percentiles
    percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
    for p in percentiles:
        stats[f'percentile_{p}'] = np.percentile(pixel_array, p)
    
    # Quartiles and IQR
    q1 = np.percentile(pixel_array, 25)
    q3 = np.percentile(pixel_array, 75)
    stats['q1'] = q1
    stats['q3'] = q3
    stats['iqr'] = q3 - q1  # Interquartile range
    
    # Distribution shape
    from scipy import stats as scipy_stats
    stats['skewness'] = scipy_stats.skew(pixel_array)
    stats['kurtosis'] = scipy_stats.kurtosis(pixel_array)
    
    # Coefficient of variation
    stats['cv'] = (stats['std'] / stats['mean']) * 100 if stats['mean'] != 0 else 0
    
    # Signal to Noise Ratio (simplified)
    stats['snr'] = stats['mean'] / stats['std'] if stats['std'] != 0 else 0
    
    return stats

def display_statistics_report(stats_dict, dicom_obj):
    """
    Display a formatted statistics report
    """
    
    print("\n" + "="*70)
    print("📊 COMPREHENSIVE STATISTICAL REPORT")
    print("="*70)
    
    print(f"\n{'IMAGE INFORMATION':^70}")
    print("-"*70)
    print(f"{'Modality:':<30} {dicom_obj.Modality}")
    print(f"{'Image Size:':<30} {dicom_obj.Rows} x {dicom_obj.Columns}")
    print(f"{'Total Pixels:':<30} {stats_dict['count']:,}")
    
    print(f"\n{'CENTRAL TENDENCY MEASURES':^70}")
    print("-"*70)
    print(f"{'Mean:':<30} {stats_dict['mean']:>20.2f}")
    print(f"{'Median:':<30} {stats_dict['median']:>20.2f}")
    print(f"{'Mode:':<30} {stats_dict['mode']:>20}")
    
    print(f"\n{'DISPERSION MEASURES':^70}")
    print("-"*70)
    print(f"{'Standard Deviation:':<30} {stats_dict['std']:>20.2f}")
    print(f"{'Variance:':<30} {stats_dict['variance']:>20.2f}")
    print(f"{'Range (Max - Min):':<30} {stats_dict['range']:>20.2f}")
    print(f"{'Interquartile Range (IQR):':<30} {stats_dict['iqr']:>20.2f}")
    print(f"{'Coefficient of Variation:':<30} {stats_dict['cv']:>19.2f}%")
    
    print(f"\n{'EXTREME VALUES':^70}")
    print("-"*70)
    print(f"{'Minimum:':<30} {stats_dict['min']:>20.2f}")
    print(f"{'Maximum:':<30} {stats_dict['max']:>20.2f}")
    print(f"{'1st Percentile:':<30} {stats_dict['percentile_1']:>20.2f}")
    print(f"{'99th Percentile:':<30} {stats_dict['percentile_99']:>20.2f}")
    
    print(f"\n{'QUARTILES':^70}")
    print("-"*70)
    print(f"{'Q1 (25th percentile):':<30} {stats_dict['q1']:>20.2f}")
    print(f"{'Q2 (50th percentile/Median):':<30} {stats_dict['median']:>20.2f}")
    print(f"{'Q3 (75th percentile):':<30} {stats_dict['q3']:>20.2f}")
    
    print(f"\n{'DISTRIBUTION SHAPE':^70}")
    print("-"*70)
    print(f"{'Skewness:':<30} {stats_dict['skewness']:>20.4f}")
    skew_interpretation = "Right-skewed" if stats_dict['skewness'] > 0 else "Left-skewed" if stats_dict['skewness'] < 0 else "Symmetric"
    print(f"{'  → Interpretation:':<30} {skew_interpretation:>20}")
    
    print(f"{'Kurtosis:':<30} {stats_dict['kurtosis']:>20.4f}")
    kurt_interpretation = "Heavy-tailed" if stats_dict['kurtosis'] > 0 else "Light-tailed" if stats_dict['kurtosis'] < 0 else "Normal"
    print(f"{'  → Interpretation:':<30} {kurt_interpretation:>20}")
    
    print(f"\n{'QUALITY METRICS':^70}")
    print("-"*70)
    print(f"{'Signal-to-Noise Ratio (SNR):':<30} {stats_dict['snr']:>20.2f}")
    
    print("\n" + "="*70)
    
    # Interpretation guide
    print(f"\n{'📖 INTERPRETATION GUIDE':^70}")
    print("-"*70)
    print("""
    Skewness:
      • = 0  : Symmetric distribution
      • > 0  : Right-skewed (tail extends right, more low values)
      • < 0  : Left-skewed (tail extends left, more high values)
    
    Kurtosis:
      • = 0  : Normal distribution (mesokurtic)
      • > 0  : Heavy-tailed, more outliers (leptokurtic)
      • < 0  : Light-tailed, fewer outliers (platykurtic)
    
    Coefficient of Variation:
      • Low (<15%) : Low variability relative to mean
      • High (>30%) : High variability relative to mean
    """)

# Perform and display statistical analysis
stats_result = comprehensive_statistical_analysis(dicom_data)
display_statistics_report(stats_result, dicom_data)


## Ενότητα 28 — Συγκριτική Οπτικοποίηση Στατιστικών

### Γιατί δεν αρκούν οι αριθμοί;

Στην προηγούμενη ενότητα παραγάγαμε δεκάδες αριθμούς. Αλλά οι άνθρωποι δεν διαβάζουμε καλά πίνακες με 30+ νούμερα — βλέπουμε πολύ καλύτερα **σχήματα και σχέσεις**. Σε αυτή την ενότητα θα δείτε τέσσερα κλασικά διαγράμματα που «μεταφράζουν» τη στατιστική σε εικόνα.

### Τα τέσσερα διαγράμματα του κώδικα

#### 1. Box plot (διάγραμμα κουτιού)
Πέντε αριθμοί σε ένα σχήμα: minimum, Q1, διάμεσος, Q3, maximum.

- Το **κουτί** καλύπτει το IQR (το μεσαίο 50% των pixel).
- Η **κόκκινη γραμμή** μέσα είναι η διάμεσος.
- Τα **whiskers** (μουστάκια) εκτείνονται μέχρι περίπου τα ακραία «κανονικά» σημεία.
- Σημεία πέρα από αυτά → **outliers**.

> Χρήση: Γρήγορη αναγνώριση ακραίων τιμών και ασυμμετρίας.

#### 2. Violin plot (διάγραμμα βιολιού)
Συνδυάζει box plot + ιστόγραμμα. Το πάχος του «βιολιού» σε κάθε ύψος δείχνει **πυκνότητα pixel**. Όπου το βιολί φαρδαίνει, εκεί συγκεντρώνονται περισσότερα pixel.

> Χρήση: Όταν θέλετε να φανεί η μορφή της κατανομής, όχι μόνο τα στατιστικά της.

#### 3. Q-Q plot (Quantile-Quantile)
**Το πιο σημαντικό διαγνωστικό διάγραμμα** της ενότητας. Συγκρίνει τα ποσοστημόρια των δεδομένων μας με αυτά μιας **κανονικής κατανομής**.

- Αν τα σημεία πέφτουν πάνω στην ευθεία αναφοράς → η κατανομή είναι περίπου κανονική.
- Αν τα σημεία αποκλίνουν, ειδικά στις άκρες → η κατανομή **δεν είναι κανονική** (συνηθισμένο για ιατρικές εικόνες, που έχουν συχνά πολυτροπικές κατανομές με φόντο, ιστούς, οστά).

> ⚠️ Πολλές στατιστικές μέθοδοι (π.χ. t-test) προϋποθέτουν κανονικότητα. Πάντα ελέγχετε με Q-Q plot πριν εφαρμόσετε.

#### 4. Bar chart κύριων στατιστικών
Ραβδόγραμμα που δείχνει σε μία ματιά τη σχέση μέσης τιμής, διαμέσου, τυπικής απόκλισης και τεταρτημορίων. Χρήσιμο για **παρουσιάσεις** και αναφορές.

> 💡 **Tip:** Όταν παρουσιάζετε αποτελέσματα σε γιατρό ή σε επιστημονική επιτροπή, το box plot και το violin plot «μιλάνε» πιο γρήγορα από τους πίνακες.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          COMPARATIVE STATISTICS VISUALIZATION                  ║
╚════════════════════════════════════════════════════════════════╝
""")

def visualize_statistics_comparison(dicom_obj):
    """
    Create visualizations comparing different statistical aspects
    """
    
    pixel_array = dicom_obj.pixel_array
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Box Plot
    axes[0, 0].boxplot(pixel_array.flatten(), vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightblue', color='blue'),
                       medianprops=dict(color='red', linewidth=2),
                       whiskerprops=dict(color='blue'),
                       capprops=dict(color='blue'))
    
    axes[0, 0].set_ylabel('Pixel Intensity', fontsize=12)
    axes[0, 0].set_title('Box Plot - Distribution Summary', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Add annotations
    q1, median, q3 = np.percentile(pixel_array.flatten(), [25, 50, 75])
    axes[0, 0].text(1.15, q1, f'Q1: {q1:.1f}', fontsize=10, va='center')
    axes[0, 0].text(1.15, median, f'Median: {median:.1f}', fontsize=10, va='center', color='red')
    axes[0, 0].text(1.15, q3, f'Q3: {q3:.1f}', fontsize=10, va='center')
    
    # 2. Violin Plot (using histogram approximation)
    axes[0, 1].violinplot([pixel_array.flatten()], vert=True, showmeans=True, showmedians=True)
    axes[0, 1].set_ylabel('Pixel Intensity', fontsize=12)
    axes[0, 1].set_title('Violin Plot - Distribution Density', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # 3. Q-Q Plot (Quantile-Quantile)
    from scipy import stats as scipy_stats
    theoretical_quantiles = scipy_stats.norm.ppf(np.linspace(0.01, 0.99, 100))
    sample_quantiles = np.percentile(pixel_array.flatten(), np.linspace(1, 99, 100))
    
    axes[1, 0].scatter(theoretical_quantiles, sample_quantiles, alpha=0.6, s=20)
    axes[1, 0].plot(theoretical_quantiles, 
                    theoretical_quantiles * pixel_array.std() + pixel_array.mean(),
                    'r--', linewidth=2, label='Normal distribution reference')
    axes[1, 0].set_xlabel('Theoretical Quantiles (Normal)', fontsize=12)
    axes[1, 0].set_ylabel('Sample Quantiles', fontsize=12)
    axes[1, 0].set_title('Q-Q Plot - Normality Check', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Statistical Summary Bar Chart
    stats_summary = {
        'Mean': pixel_array.mean(),
        'Median': np.median(pixel_array),
        'Std Dev': pixel_array.std(),
        'Q1': np.percentile(pixel_array, 25),
        'Q3': np.percentile(pixel_array, 75)
    }
    
    bars = axes[1, 1].bar(stats_summary.keys(), stats_summary.values(), 
                          color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8'],
                          edgecolor='black', linewidth=1.5)
    
    axes[1, 1].set_ylabel('Value', fontsize=12)
    axes[1, 1].set_title('Key Statistics Comparison', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                       f'{height:.1f}',
                       ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

visualize_statistics_comparison(dicom_data)


## Ενότητα 29 — Στατιστική ανά Περιοχή (Region-Based Statistics)

### Το πρόβλημα του «καθολικού μέσου όρου»

Μέχρι τώρα υπολογίζαμε στατιστικά για **όλη την εικόνα**. Αλλά μια ιατρική εικόνα είναι σπάνια ομοιόμορφη! Ένας όγκος καταλαμβάνει το 5% της εικόνας — αν τον αναμίξετε με 95% υγιή ιστό, τα στατιστικά του «πνίγονται». Επιπλέον, τεχνικά artifacts (π.χ. **ανομοιογένεια πεδίου** στο MRI, *bias field*) εμφανίζονται ως χωρικές διαφορές που η καθολική μέση τιμή κρύβει εντελώς.

**Λύση:** χωρίζουμε την εικόνα σε υπο-περιοχές και υπολογίζουμε στατιστικά **σε κάθε μία ξεχωριστά**.

### Πότε μας ενδιαφέρει η ανάλυση ανά περιοχή;

- **Έλεγχος ομοιογένειας** του φόντου (background uniformity) — βασικό QC.
- **Εντοπισμός artifacts** — αν μια περιοχή έχει πολύ διαφορετική τυπική απόκλιση από τις γειτονικές, κάτι «πειράζει».
- **Ετερογένεια όγκου** — στην ογκολογία, η χωρική ετερογένεια είναι προγνωστικός παράγοντας.
- **Bias field correction** — εντοπισμός βαθμωτών μεταβολών έντασης σε MRI.

### Τι κάνει ο κώδικας

Η `analyze_image_regions` χωρίζει την εικόνα σε **πλέγμα 3×3** (9 περιοχές) και για κάθε μία υπολογίζει mean, std, min, max, median. Το αποτέλεσμα γίνεται `pandas.DataFrame` — μια εξαιρετικά βολική μορφή για περαιτέρω ανάλυση.

Στη συνέχεια η `visualize_region_statistics` παράγει:

1. **Heatmap μέσης τιμής** — Δείχνει χωρικά πού είναι «πιο φωτεινή» η εικόνα. Ομοιόμορφο χρώμα → ομοιόμορφη εικόνα. Βαθμωτό χρώμα → bias.
2. **Heatmap τυπικής απόκλισης** — Πού υπάρχει μεγαλύτερη μεταβλητότητα (συχνά εκεί όπου υπάρχουν δομές, ακμές, ιστοί).
3. **Bar chart με error bars** — Μέση τιμή ανά περιοχή με ±1 std.
4. **CV ανά περιοχή** — Σχετική μεταβλητότητα. Βοηθά στον εντοπισμό περιοχών με ασυνήθιστα υψηλή ή χαμηλή ετερογένεια.

> 📌 **Παράμετρος `grid_size`:** Στον κώδικα είναι 3 (3×3 = 9 περιοχές). Σε εικόνες υψηλής ανάλυσης ίσως θέλετε 5×5 ή 10×10. Στις ασκήσεις θα παίξετε με αυτή την παράμετρο.

> 🧠 **Σύνδεση με προχωρημένες έννοιες:** Αυτή η ιδέα — υπολογισμός χαρακτηριστικών σε τοπικά «patches» — είναι **ο πυρήνας** πολλών σύγχρονων μεθόδων: convolutional neural networks, radiomics texture features, και sliding-window analysis.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              REGION-BASED STATISTICS                           ║
╚════════════════════════════════════════════════════════════════╝

Analyzing statistics in different regions can reveal:
✓ Spatial variations in image intensity
✓ Tissue heterogeneity
✓ Image artifacts
✓ Quality assessment
""")

def analyze_image_regions(dicom_obj, grid_size=3):
    """
    Divide image into grid and analyze each region separately
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object
    grid_size : int
        Size of the grid (e.g., 3 means 3x3 = 9 regions)
    
    Returns:
    --------
    pd.DataFrame : Statistics for each region
    """
    
    img = dicom_obj.pixel_array
    rows, cols = img.shape
    
    # Calculate region sizes
    row_step = rows // grid_size
    col_step = cols // grid_size
    
    region_stats = []
    
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(12, 12))
    
    for i in range(grid_size):
        for j in range(grid_size):
            # Extract region
            r_start = i * row_step
            r_end = (i + 1) * row_step if i < grid_size - 1 else rows
            c_start = j * col_step
            c_end = (j + 1) * col_step if j < grid_size - 1 else cols
            
            region = img[r_start:r_end, c_start:c_end]
            
            # Calculate statistics
            region_stats.append({
                'Region': f'R{i+1}C{j+1}',
                'Row': i + 1,
                'Col': j + 1,
                'Mean': region.mean(),
                'Std': region.std(),
                'Min': region.min(),
                'Max': region.max(),
                'Median': np.median(region)
            })
            
            # Visualize region
            ax = axes[i, j] if grid_size > 1 else axes
            ax.imshow(region, cmap='gray')
            ax.set_title(f'R{i+1}C{j+1}\nμ={region.mean():.0f}', fontsize=9)
            ax.axis('off')
            
            # Add colored border based on mean intensity
            for spine in ax.spines.values():
                spine.set_edgecolor('red' if region.mean() > img.mean() else 'blue')
                spine.set_linewidth(3)
    
    plt.suptitle(f'{grid_size}x{grid_size} Region Analysis\nRed=Above Average, Blue=Below Average',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Create DataFrame
    df = pd.DataFrame(region_stats)
    
    return df

# Perform region analysis
print("\nPerforming 3x3 region analysis...")
region_df = analyze_image_regions(dicom_data, grid_size=3)
print("\n📊 Region Statistics Table:")
display(region_df)

# Visualize region statistics
def visualize_region_statistics(region_df):
    """
    Visualize statistics across regions
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Mean intensity heatmap
    mean_grid = region_df.pivot(index='Row', columns='Col', values='Mean')
    im1 = axes[0, 0].imshow(mean_grid, cmap='hot', aspect='auto')
    axes[0, 0].set_title('Mean Intensity Heatmap', fontweight='bold')
    axes[0, 0].set_xlabel('Column')
    axes[0, 0].set_ylabel('Row')
    plt.colorbar(im1, ax=axes[0, 0])
    
    # Add values on heatmap
    for i in range(len(mean_grid)):
        for j in range(len(mean_grid.columns)):
            axes[0, 0].text(j, i, f'{mean_grid.iloc[i, j]:.0f}',
                          ha='center', va='center', color='white', fontweight='bold')
    
    # Standard deviation heatmap
    std_grid = region_df.pivot(index='Row', columns='Col', values='Std')
    im2 = axes[0, 1].imshow(std_grid, cmap='viridis', aspect='auto')
    axes[0, 1].set_title('Standard Deviation Heatmap', fontweight='bold')
    axes[0, 1].set_xlabel('Column')
    axes[0, 1].set_ylabel('Row')
    plt.colorbar(im2, ax=axes[0, 1])
    
    # Add values on heatmap
    for i in range(len(std_grid)):
        for j in range(len(std_grid.columns)):
            axes[0, 1].text(j, i, f'{std_grid.iloc[i, j]:.0f}',
                          ha='center', va='center', color='white', fontweight='bold')
    
    # Bar chart comparison
    x_pos = np.arange(len(region_df))
    axes[1, 0].bar(x_pos, region_df['Mean'], alpha=0.7, label='Mean', color='steelblue')
    axes[1, 0].errorbar(x_pos, region_df['Mean'], yerr=region_df['Std'], 
                       fmt='none', color='red', capsize=5, label='±1 Std')
    axes[1, 0].set_xlabel('Region')
    axes[1, 0].set_ylabel('Intensity')
    axes[1, 0].set_title('Mean Intensity per Region (with Std Dev)', fontweight='bold')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(region_df['Region'], rotation=45)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Coefficient of variation
    cv = (region_df['Std'] / region_df['Mean']) * 100
    axes[1, 1].bar(region_df['Region'], cv, color='coral', edgecolor='black')
    axes[1, 1].set_xlabel('Region')
    axes[1, 1].set_ylabel('Coefficient of Variation (%)')
    axes[1, 1].set_title('Variability per Region', fontweight='bold')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].axhline(y=cv.mean(), color='red', linestyle='--', 
                      linewidth=2, label=f'Mean CV: {cv.mean():.1f}%')
    axes[1, 1].legend()
    
    plt.tight_layout()
    plt.show()

visualize_region_statistics(region_df)


## Ενότητα 30 — Ανάλυση Προφίλ Έντασης (Intensity Profiles)

### Τι είναι ένα προφίλ έντασης;

Φανταστείτε ότι «κόβετε» την εικόνα κατά μήκος μιας ευθείας γραμμής και κοιτάτε τις τιμές των pixel πάνω σε αυτή τη γραμμή. Αυτό είναι ένα **προφίλ έντασης**: μια μονοδιάστατη ακολουθία αριθμών που δείχνει πώς αλλάζει η ένταση κατά μήκος της γραμμής.

> 📌 **Αναλογία:** Όπως όταν ένα GPS δείχνει το **υψομετρικό προφίλ** μιας πεζοπορίας — από επίπεδο σε ανήφορο, σε κορυφή, σε κατήφορο. Εδώ το «υψόμετρο» είναι η ένταση των pixel.

### Γιατί είναι χρήσιμα;

1. **Εντοπισμός ορίων (edges):** Όπου το προφίλ αλλάζει απότομα → εκεί υπάρχει σύνορο μεταξύ ιστών.
2. **Μέτρηση αποστάσεων:** Συνδυάζοντας με το `PixelSpacing` του DICOM, μπορούμε να μετρήσουμε διαστάσεις ανατομικών δομών σε χιλιοστά.
3. **Quality control:** Αν το προφίλ φαίνεται «θολωμένο» αντί για απότομες μεταβάσεις, η εικόνα έχει χαμηλή χωρική ανάλυση ή κίνηση.
4. **Ποσοτική σύγκριση:** Πριν/μετά θεραπεία — πώς άλλαξε το προφίλ στην ίδια θέση.

### Τι κάνει ο κώδικας

Η `analyze_intensity_profiles` εξάγει **τρία προφίλ**:

- **Οριζόντιο** — η μεσαία γραμμή της εικόνας.
- **Κάθετο** — η μεσαία στήλη.
- **Διαγώνιο** — από επάνω-αριστερά προς κάτω-δεξιά.

Και στη συνέχεια υπολογίζει την **κλίση (gradient)** του οριζόντιου προφίλ. Αυτό είναι κρίσιμο: η κλίση είναι ο **ρυθμός μεταβολής** της έντασης. Όταν η κλίση είναι κοντά στο 0, βρισκόμαστε σε μια ομοιογενή περιοχή. Όταν η κλίση εκτοξεύεται (θετικά ή αρνητικά), έχουμε **ακμή** — μετάβαση από έναν ιστό σε άλλον.

```
Προφίλ:    ─────╱╲────────╱╲──────
Gradient:  ─0─▲──▼─0─0─▲──▼─0──
                ↑       ↑
              ακμές (edges)
```

> 🧠 **Σύνδεση με computer vision:** Όλοι οι κλασικοί ανιχνευτές ακμών (Sobel, Canny) βασίζονται στην ιδέα της κλίσης. Εδώ τη βλέπετε στην απλούστερη μορφή της — σε μία διάσταση.

> 💡 **Στην κλινική πράξη:** Οι ακτινολόγοι σχεδιάζουν προφίλ για να μετρήσουν π.χ. το πλάτος αρτηρίας ή τη διάμετρο όγκου με ακρίβεια — όχι «με το μάτι».


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              INTENSITY PROFILE ANALYSIS                        ║
╚════════════════════════════════════════════════════════════════╝

Intensity profiles show how pixel values change along a line.
Useful for:
✓ Measuring edges and boundaries
✓ Detecting gradients
✓ Quality control
✓ Quantitative analysis
""")

def analyze_intensity_profiles(dicom_obj):
    """
    Analyze intensity profiles along different directions
    """
    
    img = dicom_obj.pixel_array
    rows, cols = img.shape
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    
    # Show image with profile lines
    axes[0, 0].imshow(img, cmap='gray')
    axes[0, 0].set_title('Image with Profile Lines', fontweight='bold')
    
    # Horizontal profile (middle row)
    mid_row = rows // 2
    axes[0, 0].axhline(y=mid_row, color='red', linestyle='--', linewidth=2, label='Horizontal')
    profile_h = img[mid_row, :]
    
    axes[0, 1].plot(profile_h, color='red', linewidth=2)
    axes[0, 1].set_title(f'Horizontal Profile (Row {mid_row})', fontweight='bold')
    axes[0, 1].set_xlabel('Column Position')
    axes[0, 1].set_ylabel('Intensity')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].fill_between(range(len(profile_h)), profile_h, alpha=0.3, color='red')
    
    # Vertical profile (middle column)
    mid_col = cols // 2
    axes[0, 0].axvline(x=mid_col, color='blue', linestyle='--', linewidth=2, label='Vertical')
    profile_v = img[:, mid_col]
    
    axes[1, 0].plot(profile_v, color='blue', linewidth=2)
    axes[1, 0].set_title(f'Vertical Profile (Column {mid_col})', fontweight='bold')
    axes[1, 0].set_xlabel('Row Position')
    axes[1, 0].set_ylabel('Intensity')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].fill_between(range(len(profile_v)), profile_v, alpha=0.3, color='blue')
    
    # Diagonal profile (top-left to bottom-right)
    diag_length = min(rows, cols)
    diag_indices = np.linspace(0, diag_length-1, diag_length).astype(int)
    profile_diag = img[diag_indices, diag_indices]
    
    axes[0, 0].plot([0, diag_length-1], [0, diag_length-1], 
                    color='green', linestyle='--', linewidth=2, label='Diagonal')
    axes[0, 0].legend()
    
    axes[1, 1].plot(profile_diag, color='green', linewidth=2)
    axes[1, 1].set_title('Diagonal Profile (TL to BR)', fontweight='bold')
    axes[1, 1].set_xlabel('Position along Diagonal')
    axes[1, 1].set_ylabel('Intensity')
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].fill_between(range(len(profile_diag)), profile_diag, alpha=0.3, color='green')
    
    # Profile statistics comparison
    profiles_data = {
        'Horizontal': profile_h,
        'Vertical': profile_v,
        'Diagonal': profile_diag
    }
    
    profile_stats = []
    for name, profile in profiles_data.items():
        profile_stats.append({
            'Profile': name,
            'Mean': profile.mean(),
            'Std': profile.std(),
            'Min': profile.min(),
            'Max': profile.max(),
            'Range': profile.max() - profile.min()
        })
    
    profile_stats_df = pd.DataFrame(profile_stats)
    
    # Plot statistics comparison
    x = np.arange(len(profile_stats_df))
    width = 0.15
    
    axes[2, 0].bar(x - 2*width, profile_stats_df['Mean'], width, label='Mean', color='steelblue')
    axes[2, 0].bar(x - width, profile_stats_df['Std'], width, label='Std', color='orange')
    axes[2, 0].bar(x, profile_stats_df['Min'], width, label='Min', color='green')
    axes[2, 0].bar(x + width, profile_stats_df['Max'], width, label='Max', color='red')
    axes[2, 0].bar(x + 2*width, profile_stats_df['Range'], width, label='Range', color='purple')
    
    axes[2, 0].set_xlabel('Profile Type')
    axes[2, 0].set_ylabel('Value')
    axes[2, 0].set_title('Profile Statistics Comparison', fontweight='bold')
    axes[2, 0].set_xticks(x)
    axes[2, 0].set_xticklabels(profile_stats_df['Profile'])
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3, axis='y')
    
    # Gradient analysis (rate of change)
    gradient_h = np.gradient(profile_h)
    axes[2, 1].plot(gradient_h, color='darkred', linewidth=2)
    axes[2, 1].set_title('Horizontal Profile Gradient (Rate of Change)', fontweight='bold')
    axes[2, 1].set_xlabel('Column Position')
    axes[2, 1].set_ylabel('Gradient (Intensity Change)')
    axes[2, 1].grid(True, alpha=0.3)
    axes[2, 1].axhline(y=0, color='black', linestyle='-', linewidth=1)
    
    plt.tight_layout()
    plt.show()
    
    return profile_stats_df

# Perform profile analysis
profile_stats = analyze_intensity_profiles(dicom_data)
print("\n📊 Profile Statistics:")
display(profile_stats)


## Ενότητα 31 — Εξίσωση Ιστογράμματος (Histogram Equalization)

### Το πρόβλημα της χαμηλής αντίθεσης

Πολλές ιατρικές εικόνες χρησιμοποιούν **μόνο ένα μικρό κομμάτι** του διαθέσιμου εύρους έντασης. Αν π.χ. μια εικόνα 8-bit (0–255) χρησιμοποιεί μόνο τιμές 80–140, το ιστόγραμμα είναι «στριμωγμένο» στη μέση και η εικόνα φαίνεται **γκρίζα και άτονη** — χάνουμε λεπτομέρειες.

### Η ιδέα της εξίσωσης

Η **εξίσωση ιστογράμματος** είναι μια μη-γραμμική μεταμόρφωση που **ξαπλώνει** την κατανομή των εντάσεων ώστε να καλύψει όλο το διαθέσιμο εύρος. Στόχος: ένα ιστόγραμμα όσο πιο **ομοιόμορφο** γίνεται, που μεταφράζεται σε εικόνα με αυξημένη αντίθεση.

### Πώς δουλεύει — απλή μαθηματική διαίσθηση

Το κλειδί είναι η **CDF** (Cumulative Distribution Function): η αθροιστική κατανομή των εντάσεων. Τα βήματα είναι:

1. Υπολογίζουμε το ιστόγραμμα `h(i)`.
2. Υπολογίζουμε την CDF: `cdf(i) = Σ h(j) για j ≤ i`.
3. **Κανονικοποιούμε** την CDF στο εύρος [0, 255].
4. Χρησιμοποιούμε την κανονικοποιημένη CDF ως **lookup table**: κάθε αρχικό pixel τιμής `i` αντικαθίσταται με την τιμή `cdf'(i)`.

> 🧠 **Γιατί δουλεύει:** Η CDF, όταν κανονικοποιηθεί, είναι ο «τέλειος» μετασχηματισμός που μετατρέπει οποιαδήποτε κατανομή σε ομοιόμορφη.

### Τι θα δείτε στον κώδικα

Έξι υποδιαγράμματα που δείχνουν όλη τη διαδικασία:

| # | Τι δείχνει | Τι μαθαίνετε |
|---|------------|--------------|
| 1 | Αρχική εικόνα | Σημείο εκκίνησης |
| 2 | Αρχικό ιστόγραμμα | Πόσο «στριμωγμένο» ήταν |
| 3 | CDF | Η συνάρτηση μετασχηματισμού |
| 4 | Εικόνα μετά την εξίσωση | Αυξημένη αντίθεση |
| 5 | Νέο ιστόγραμμα | Πιο ομοιόμορφη κατανομή |
| 6 | Mapping function | Πώς απεικονίζονται οι παλιές → νέες τιμές |

### ⚠️ Προσοχή στην ιατρική χρήση

Η εξίσωση ιστογράμματος **αλλάζει** τις απόλυτες τιμές των pixel. Αυτό σημαίνει ότι:

- **ΔΕΝ** μπορεί να εφαρμοστεί σε CT αν θέλετε να διατηρήσετε τις **Hounsfield Units** (που έχουν φυσικό νόημα — π.χ. −1000 = αέρας, 0 = νερό).
- Είναι κατάλληλη για **οπτικοποίηση** ή ως pre-processing σε νευρωνικά δίκτυα φυσικών εικόνων.
- Σε MRI, μπορεί να χρησιμοποιηθεί προσεκτικά αν δεν σας ενδιαφέρουν απόλυτες εντάσεις.

> 📌 **Εναλλακτική:** Το **CLAHE** (Contrast-Limited Adaptive Histogram Equalization) εφαρμόζει εξίσωση τοπικά και είναι πιο φιλικό σε ιατρικές εικόνες. Δείτε το `cv2.createCLAHE()` αν θέλετε να πειραματιστείτε.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              HISTOGRAM EQUALIZATION                            ║
╚════════════════════════════════════════════════════════════════╝

Histogram equalization is a technique to improve image contrast
by redistributing pixel intensities to use the full dynamic range.

Applications:
✓ Enhancing low-contrast images
✓ Improving visibility of details
✓ Preprocessing for analysis
""")

def demonstrate_histogram_equalization(dicom_obj):
    """
    Demonstrate histogram equalization
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    # Normalize to 0-255 for equalization
    img_normalized = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    
    # Compute histogram
    hist, bins = np.histogram(img_normalized.flatten(), bins=256, range=[0, 256])
    
    # Compute cumulative distribution function (CDF)
    cdf = hist.cumsum()
    cdf_normalized = cdf * hist.max() / cdf.max()  # Normalize for display
    
    # Equalization
    cdf_masked = np.ma.masked_equal(cdf, 0)
    cdf_masked = (cdf_masked - cdf_masked.min()) * 255 / (cdf_masked.max() - cdf_masked.min())
    cdf_final = np.ma.filled(cdf_masked, 0).astype('uint8')
    
    # Apply equalization
    img_equalized = cdf_final[img_normalized]
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    # Original image
    axes[0, 0].imshow(img_normalized, cmap='gray')
    axes[0, 0].set_title('Original Image', fontweight='bold', fontsize=12)
    axes[0, 0].axis('off')
    
    # Original histogram
    axes[0, 1].hist(img_normalized.flatten(), bins=256, range=[0, 256], 
                    color='steelblue', alpha=0.7, edgecolor='black')
    axes[0, 1].set_title('Original Histogram', fontweight='bold', fontsize=12)
    axes[0, 1].set_xlabel('Pixel Intensity')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].grid(True, alpha=0.3)
    
    # CDF
    axes[0, 2].plot(cdf_normalized, color='darkblue', linewidth=2)
    axes[0, 2].set_title('Cumulative Distribution Function', fontweight='bold', fontsize=12)
    axes[0, 2].set_xlabel('Pixel Intensity')
    axes[0, 2].set_ylabel('Cumulative Frequency')
    axes[0, 2].grid(True, alpha=0.3)
    axes[0, 2].set_xlim([0, 256])
    
    # Equalized image
    axes[1, 0].imshow(img_equalized, cmap='gray')
    axes[1, 0].set_title('Equalized Image', fontweight='bold', fontsize=12)
    axes[1, 0].axis('off')
    
    # Equalized histogram
    axes[1, 1].hist(img_equalized.flatten(), bins=256, range=[0, 256],
                    color='coral', alpha=0.7, edgecolor='black')
    axes[1, 1].set_title('Equalized Histogram', fontweight='bold', fontsize=12)
    axes[1, 1].set_xlabel('Pixel Intensity')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Comparison
    axes[1, 2].plot(cdf_final, color='red', linewidth=2, label='Mapping Function')
    axes[1, 2].set_title('Equalization Mapping Function', fontweight='bold', fontsize=12)
    axes[1, 2].set_xlabel('Input Intensity')
    axes[1, 2].set_ylabel('Output Intensity')
    axes[1, 2].grid(True, alpha=0.3)
    axes[1, 2].legend()
    axes[1, 2].set_xlim([0, 256])
    axes[1, 2].set_ylim([0, 256])
    
    plt.suptitle('Histogram Equalization Process', fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()
    
    # Statistics comparison
    print("\n📊 Comparison Statistics:")
    print("="*60)
    print(f"{'Metric':<30} {'Original':>12} {'Equalized':>12}")
    print("-"*60)
    print(f"{'Mean':<30} {img_normalized.mean():>12.2f} {img_equalized.mean():>12.2f}")
    print(f"{'Std Dev':<30} {img_normalized.std():>12.2f} {img_equalized.std():>12.2f}")
    print(f"{'Min':<30} {img_normalized.min():>12.2f} {img_equalized.min():>12.2f}")
    print(f"{'Max':<30} {img_normalized.max():>12.2f} {img_equalized.max():>12.2f}")
    print(f"{'Dynamic Range':<30} {img_normalized.ptp():>12.2f} {img_equalized.ptp():>12.2f}")
    print("="*60)

demonstrate_histogram_equalization(dicom_data)


## Ενότητα 32 — Ανάλυση Αντίθεσης και Φωτεινότητας

### Δύο διαφορετικές έννοιες — μη τις μπερδεύετε

Στην καθημερινή ομιλία λέμε «αυτή η εικόνα είναι σκοτεινή» ή «έχει χαμηλή αντίθεση» χωρίς πολλή σκέψη. Στην ποσοτική ανάλυση όμως είναι **δύο εντελώς ξεχωριστές μετρικές**:

| Έννοια | Τι μετρά | Σχέση με στατιστική |
|--------|----------|----------------------|
| **Φωτεινότητα** (brightness) | Συνολικό «επίπεδο» της εικόνας | Μέση τιμή των pixel |
| **Αντίθεση** (contrast) | Διαφορά μεταξύ φωτεινών και σκοτεινών περιοχών | Διασπορά των pixel |

> 💡 **Φανταστείτε:** Δύο εικόνες με την **ίδια** μέση τιμή. Η μία έχει pixel στην περιοχή 100–110 (πολύ χαμηλή αντίθεση — όλα μοιάζουν). Η άλλη έχει pixel σε όλη την κλίμακα 0–255 (υψηλή αντίθεση). Ίδια φωτεινότητα, εντελώς διαφορετική αντίθεση.

### Δύο τρόποι να μετρήσετε αντίθεση

Στον κώδικα υπολογίζονται και οι δύο γνωστές μετρικές:

#### 1. RMS contrast
$$
C_{\text{RMS}} = \sigma_I
$$
Απλή τυπική απόκλιση (στην κανονικοποιημένη εικόνα). **Δίνει βάρος σε όλα τα pixel** και είναι πιο αντιπροσωπευτική για φυσικές/ιατρικές εικόνες.

#### 2. Michelson contrast
$$
C_{\text{Michelson}} = \dfrac{I_{\max} - I_{\min}}{I_{\max} + I_{\min}}
$$
Λαμβάνει υπόψη **μόνο** τις ακραίες τιμές. Παραδοσιακά χρησιμοποιείται για **περιοδικά** μοτίβα (π.χ. test patterns σε οπτική). Σε ιατρικές εικόνες είναι ευαίσθητο σε outliers.

### Τι κάνει ο κώδικας

Δημιουργεί **πέντε εκδοχές** της ίδιας εικόνας για να δείτε **οπτικά** τη διαφορά:

1. **Αρχική** — αναφορά.
2. **Αυξημένη φωτεινότητα** — `image + 0.2`. Όλα τα pixel ανεβαίνουν.
3. **Μειωμένη φωτεινότητα** — `image - 0.2`. Όλα τα pixel πέφτουν.
4. **Αυξημένη αντίθεση** — `(image - 0.5) * 1.5 + 0.5`. Τα pixel «τραβιούνται» μακριά από το κέντρο.
5. **Μειωμένη αντίθεση** — `(image - 0.5) * 0.5 + 0.5`. Τα pixel «συμπιέζονται» γύρω από το κέντρο.

Στους τίτλους εμφανίζονται οι **νέες** τιμές brightness και contrast, ώστε να δείτε ποσοτικά πώς αλλάζουν.

> 📌 **Σύνδεση με κλινική πρακτική:** Αυτό ακριβώς που κάνει ένας ακτινολόγος όταν ρυθμίζει το **window/level** σε έναν σταθμό εργασίας — αλλάζει εικονικά την αντίθεση και τη φωτεινότητα **χωρίς να πειράζει τα δεδομένα**, για καλύτερη οπτική ερμηνεία.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          CONTRAST AND BRIGHTNESS ANALYSIS                      ║
╚════════════════════════════════════════════════════════════════╝

Understanding contrast and brightness:
- Brightness: Overall intensity level (related to mean)
- Contrast: Difference between light and dark areas (related to std dev)
""")

def analyze_contrast_brightness(dicom_obj):
    """
    Analyze and visualize contrast and brightness
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    # Normalize
    img_norm = (img - img.min()) / (img.max() - img.min())
    
    # Calculate metrics
    brightness = img_norm.mean()
    contrast = img_norm.std()
    
    # Michelson contrast (for images with distinct bright and dark regions)
    max_val = img_norm.max()
    min_val = img_norm.min()
    michelson_contrast = (max_val - min_val) / (max_val + min_val) if (max_val + min_val) > 0 else 0
    
    # RMS contrast
    rms_contrast = img_norm.std()
    
    # Create variations
    # Increase brightness
    img_bright = np.clip(img_norm + 0.2, 0, 1)
    
    # Decrease brightness
    img_dark = np.clip(img_norm - 0.2, 0, 1)
    
    # Increase contrast
    img_high_contrast = np.clip((img_norm - 0.5) * 1.5 + 0.5, 0, 1)
    
    # Decrease contrast
    img_low_contrast = np.clip((img_norm - 0.5) * 0.5 + 0.5, 0, 1)
    
    # Visualize
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    images = [
        (img_norm, 'Original'),
        (img_bright, 'Increased Brightness'),
        (img_dark, 'Decreased Brightness'),
        (img_high_contrast, 'Increased Contrast'),
        (img_low_contrast, 'Decreased Contrast'),
    ]
    
    for idx, (image, title) in enumerate(images):
        row = idx // 3
        col = idx % 3
        
        axes[row, col].imshow(image, cmap='gray')
        
        # Calculate metrics for each variation
        b = image.mean()
        c = image.std()
        
        axes[row, col].set_title(f'{title}\nBrightness: {b:.3f}, Contrast: {c:.3f}',
                                fontweight='bold', fontsize=10)
        axes[row, col].axis('off')
    
    # Add metrics display
    axes[1, 2].axis('off')
    metrics_text = f"""
    CONTRAST METRICS
    ─────────────────────
    
    Brightness (Mean):
      {brightness:.4f}
    
    RMS Contrast (Std):
      {rms_contrast:.4f}
    
    Michelson Contrast:
      {michelson_contrast:.4f}
    
    Dynamic Range:
      {img_norm.ptp():.4f}
    
    ─────────────────────
    Interpretation:
    • Brightness: [0-1]
      0.5 = medium
    • Contrast: [0-∞]
      Higher = more contrast
    • Michelson: [0-1]
      1 = maximum contrast
    """
    
    axes[1, 2].text(0.1, 0.5, metrics_text, fontsize=10, family='monospace',
                   verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Contrast and Brightness Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

analyze_contrast_brightness(dicom_data)


## Ενότητα 33 — Ανάλυση Θορύβου

### Τι είναι ο θόρυβος και από πού προέρχεται;

**Θόρυβος** είναι η ανεπιθύμητη τυχαία διακύμανση των τιμών των pixel που **δεν** σχετίζεται με την υποκείμενη ανατομία. Σε ιατρικές εικόνες προέρχεται από:

- **Θερμικός θόρυβος** — από τα ηλεκτρονικά κυκλώματα.
- **Κβαντικός θόρυβος (Poisson)** — εγγενής στη φύση των φωτονίων (CT, mammography).
- **Θόρυβος ψηφιοποίησης** — όρια ακρίβειας του ADC.
- **Κίνηση ασθενούς** — δεν είναι «θόρυβος» τεχνικά, αλλά εμφανίζεται με παρόμοιο τρόπο.

### Γιατί μας ενδιαφέρει;

- **Διαγνωστική ακρίβεια:** Πολύς θόρυβος καλύπτει μικρές βλάβες.
- **Αυτόματη ανάλυση:** Αλγόριθμοι segmentation, registration και AI παίρνουν χειρότερα αποτελέσματα σε θορυβώδεις εικόνες.
- **Quality assurance:** Σε κλινικά μηχανήματα, ο τακτικός έλεγχος SNR είναι μέρος του πρωτοκόλλου ποιότητας.

### Τρεις μέθοδοι εκτίμησης θορύβου

#### Μέθοδος 1: Τυπική απόκλιση σε ομοιογενή περιοχή
Σε μια **καθαρή** περιοχή (π.χ. αέρας στις γωνίες) η μόνη πηγή μεταβλητότητας είναι ο θόρυβος. Άρα:
$$
\sigma_{\text{noise}} \approx \sigma_{\text{corner}}
$$
Ο κώδικας παίρνει και τις τέσσερις γωνίες, υπολογίζει τη std σε καθεμιά, και κρατά τη μέση τιμή.

> ⚠️ **Προσοχή:** Αν τυχόν μια γωνία ΔΕΝ είναι φόντο (π.χ. ένας ώμος ασθενούς πέφτει στη γωνία), η μέθοδος δίνει υπερεκτίμηση. Γι' αυτό βλέπετε ότι κρατάμε και την **ελάχιστη** corner std — πιο ανθεκτική στην εξαίρεση.

#### Μέθοδος 2: MAD (Median Absolute Deviation)
$$
\text{MAD} = \text{median}(|x - \text{median}(x)|)
$$
Πλεονέκτημα: εξαιρετικά **ανθεκτική** σε outliers. Για κανονική κατανομή, η σχέση με την τυπική απόκλιση είναι:
$$
\sigma \approx 1{,}4826 \times \text{MAD}
$$

#### Μέθοδος 3: Gradient-based
Ο θόρυβος εμφανίζεται ως υψηλο-συχνοτικές διακυμάνσεις. Άρα η τυπική απόκλιση των διαφορών μεταξύ γειτονικών pixel (`np.diff`) σχετίζεται άμεσα με το επίπεδο θορύβου.

### Το SNR και η κλίμακα ποιότητας

$$
\text{SNR} = \dfrac{\mu_{\text{signal}}}{\sigma_{\text{noise}}}, \quad \text{SNR}_{\text{dB}} = 20 \log_{10}(\text{SNR})
$$

Σε **decibel** (dB) η κλίμακα γίνεται πιο διαισθητική:

| SNR (dB) | Ποιότητα |
|----------|----------|
| < 10 dB | Φτωχή — δύσκολα διαγνωστικά |
| 10–20 dB | Αποδεκτή για βασική απεικόνιση |
| 20–30 dB | Καλή — διαγνωστική ποιότητα |
| > 30 dB | Άριστη — ιδανική για ποσοτική ανάλυση |

### Τι παράγει ο κώδικας

Έξι οπτικοποιήσεις:

1. **Αρχική εικόνα** — αναφορά.
2. **Heatmap κλίσης (gradient magnitude)** — εκεί όπου εμφανίζεται **ταυτόχρονα** θόρυβος και πραγματικές ακμές.
3. **Ιστόγραμμα κλίσεων** σε λογαριθμική κλίμακα — η ουρά του υποδηλώνει επίπεδο θορύβου.
4. **Εικόνα με σημειωμένες τις γωνίες** που χρησιμοποιήθηκαν για εκτίμηση.
5. **Πίνακας μετρικών** — όλες οι μετρήσεις σε ένα μέρος.
6. **Διάγραμμα ποιότητας** — που πέφτει η εικόνα μας στην παραπάνω κλίμακα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                    NOISE ANALYSIS                              ║
╚════════════════════════════════════════════════════════════════╝

Noise in medical images affects:
✓ Image quality
✓ Diagnostic accuracy
✓ Automated analysis algorithms

Common types:
- Gaussian noise
- Salt & pepper noise
- Poisson noise
""")

def estimate_noise_level(dicom_obj, method='std_uniform_region'):
    """
    Estimate noise level in the image
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object
    method : str
        Method for noise estimation
    
    Returns:
    --------
    dict : Noise metrics
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    noise_metrics = {}
    
    # Method 1: Standard deviation in a uniform region (background)
    # Assume corners are relatively uniform background
    corner_size = min(img.shape) // 10
    corners = [
        img[:corner_size, :corner_size],  # Top-left
        img[:corner_size, -corner_size:],  # Top-right
        img[-corner_size:, :corner_size],  # Bottom-left
        img[-corner_size:, -corner_size:]  # Bottom-right
    ]
    
    corner_stds = [corner.std() for corner in corners]
    noise_metrics['corner_std_mean'] = np.mean(corner_stds)
    noise_metrics['corner_std_min'] = np.min(corner_stds)
    
    # Method 2: Median Absolute Deviation (MAD) - robust to outliers
    median = np.median(img)
    mad = np.median(np.abs(img - median))
    noise_metrics['mad'] = mad
    noise_metrics['estimated_std_from_mad'] = 1.4826 * mad  # Conversion factor for normal distribution
    
    # Method 3: Gradient-based noise estimation
    # Noise often shows up in high-frequency components
    grad_x = np.abs(np.diff(img, axis=1))
    grad_y = np.abs(np.diff(img, axis=0))
    noise_metrics['gradient_std_x'] = grad_x.std()
    noise_metrics['gradient_std_y'] = grad_y.std()
    
    # Signal-to-Noise Ratio
    signal = img.mean()
    noise = noise_metrics['estimated_std_from_mad']
    noise_metrics['snr'] = signal / noise if noise > 0 else float('inf')
    noise_metrics['snr_db'] = 20 * np.log10(noise_metrics['snr']) if noise_metrics['snr'] > 0 else float('inf')
    
    return noise_metrics

def visualize_noise_analysis(dicom_obj):
    """
    Visualize noise characteristics
    """
    
    img = dicom_obj.pixel_array.astype(float)
    
    # Estimate noise
    noise_metrics = estimate_noise_level(dicom_obj)
    
    # Compute gradients
    grad_x = np.diff(img, axis=1)
    grad_y = np.diff(img, axis=0)
    grad_magnitude = np.sqrt(grad_x[:, :-1]**2 + grad_y[:-1, :]**2)
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # Original image
    axes[0, 0].imshow(img, cmap='gray')
    axes[0, 0].set_title('Original Image', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Gradient magnitude (edge/noise)
    im1 = axes[0, 1].imshow(grad_magnitude, cmap='hot')
    axes[0, 1].set_title('Gradient Magnitude\n(Edges + Noise)', fontweight='bold')
    axes[0, 1].axis('off')
    plt.colorbar(im1, ax=axes[0, 1])
    
    # Histogram of gradients
    axes[0, 2].hist(grad_magnitude.flatten(), bins=50, color='orange', alpha=0.7, edgecolor='black')
    axes[0, 2].set_title('Gradient Histogram', fontweight='bold')
    axes[0, 2].set_xlabel('Gradient Magnitude')
    axes[0, 2].set_ylabel('Frequency')
    axes[0, 2].set_yscale('log')
    axes[0, 2].grid(True, alpha=0.3)
    
    # Corner regions for noise estimation
    corner_size = min(img.shape) // 10
    img_corners = img.copy()
    
    # Mark corners
    from matplotlib.patches import Rectangle
    axes[1, 0].imshow(img, cmap='gray')
    corners_coords = [
        (0, 0), (img.shape[1]-corner_size, 0),
        (0, img.shape[0]-corner_size), (img.shape[1]-corner_size, img.shape[0]-corner_size)
    ]
    
    for x, y in corners_coords:
        rect = Rectangle((x, y), corner_size, corner_size, 
                        linewidth=2, edgecolor='red', facecolor='none')
        axes[1, 0].add_patch(rect)
    
    axes[1, 0].set_title('Corner Regions\n(for noise estimation)', fontweight='bold')
    axes[1, 0].axis('off')
    
    # Noise metrics display
    axes[1, 1].axis('off')
    metrics_text = f"""
    NOISE METRICS
    ══════════════════════════
    
    Corner STD (mean):
      {noise_metrics['corner_std_mean']:.2f}
    
    MAD (Median Abs Dev):
      {noise_metrics['mad']:.2f}
    
    Estimated Noise (σ):
      {noise_metrics['estimated_std_from_mad']:.2f}
    
    Gradient STD (X):
      {noise_metrics['gradient_std_x']:.2f}
    
    Gradient STD (Y):
      {noise_metrics['gradient_std_y']:.2f}
    
    Signal-to-Noise Ratio:
      {noise_metrics['snr']:.2f}
    
    SNR (dB):
      {noise_metrics['snr_db']:.2f} dB
    
    ══════════════════════════
    Higher SNR = Better quality
    Typical good SNR > 20 dB
    """
    
    axes[1, 1].text(0.1, 0.5, metrics_text, fontsize=9, family='monospace',
                   verticalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    
    # SNR visualization
    snr_categories = ['Poor\n(<10 dB)', 'Fair\n(10-20 dB)', 'Good\n(20-30 dB)', 'Excellent\n(>30 dB)']
    snr_values = [10, 20, 30, 40]
    colors = ['red', 'orange', 'yellow', 'green']
    
    bars = axes[1, 2].barh(snr_categories, snr_values, color=colors, alpha=0.6, edgecolor='black')
    
    # Mark current SNR
    current_snr = noise_metrics['snr_db']
    if current_snr < 10:
        marker_pos = 0
    elif current_snr < 20:
        marker_pos = 1
    elif current_snr < 30:
        marker_pos = 2
    else:
        marker_pos = 3
    
    axes[1, 2].plot([current_snr], [marker_pos], 'r*', markersize=20, 
                   label=f'Current: {current_snr:.1f} dB')
    
    axes[1, 2].set_xlabel('SNR (dB)')
    axes[1, 2].set_title('SNR Quality Assessment', fontweight='bold')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3, axis='x')
    
    plt.suptitle('Comprehensive Noise Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_noise_analysis(dicom_data)


## Ενότητα 34 — Σύνταξη Ολοκληρωμένης Στατιστικής Αναφοράς

### Από τα κομμάτια στο σύνολο

Μέχρι εδώ, μάθαμε **μεμονωμένες** τεχνικές: ιστογράμματα, στατιστικά, χωρική ανάλυση, θόρυβο. Σε πραγματική κλινική ή ερευνητική χρήση πρέπει να τα συνθέσουμε όλα σε **μία αναφορά** που:

- Συγκεντρώνει τα μεταδεδομένα του ασθενούς και της εξέτασης.
- Δίνει στατιστικά χαρακτηριστικά της εικόνας.
- Αξιολογεί την ποιότητα.
- Παρουσιάζει τα παραπάνω σε αναγνώσιμη μορφή για διαφορετικά ακροατήρια (γιατρός, ερευνητής, μηχανικός).

### Τι κάνει η `generate_complete_dicom_report`

Επιστρέφει ένα **nested dictionary** με τέσσερα τμήματα:

1. **`metadata`** — Modality, Patient ID, Study Date, μέγεθος εικόνας, pixel spacing, slice thickness.
2. **`basic_stats`** — Όλα τα στατιστικά της Ενότητας 27.
3. **`histogram_stats`** — Skewness, kurtosis, CV.
4. **`quality_metrics`** — SNR, εκτίμηση θορύβου από MAD.
5. **`spatial_stats`** — Brightness, RMS contrast, dynamic range.

Ταυτόχρονα παράγει **ένα ολοκληρωμένο γραφικό dashboard** με:

- Την εικόνα.
- Το ιστόγραμμα με σημειωμένη μέση τιμή και διάμεσο.
- Box plot.
- Πίνακες μεταδεδομένων, στατιστικών και ποιότητας.
- Προφίλ έντασης μέσης γραμμής.

> 💡 **Καλή πρακτική κώδικα:** Παρατηρήστε ότι η συνάρτηση δέχεται προαιρετικό `save_path`. Έτσι η ίδια λειτουργία μπορεί να (α) εμφανίσει την αναφορά στο Jupyter για εξερεύνηση, ή (β) να την αποθηκεύσει σε αρχείο για παράδοση. Αυτό λέγεται **separation of concerns** και είναι θεμελιώδες σε επαγγελματικό κώδικα.

### Γιατί μας νοιάζει το JSON output

Στο τέλος του κελιού βλέπετε `print(json.dumps(report_data, indent=2))`. Γιατί JSON;

- **Διαλειτουργικότητα:** Άλλα προγράμματα μπορούν να διαβάσουν το JSON και να επεξεργαστούν τα αποτελέσματα.
- **Αρχειοθέτηση:** Σε μελέτες με 1000+ εικόνες, αποθηκεύουμε ένα JSON ανά εικόνα και τα συγκεντρώνουμε σε database.
- **Reproducibility:** Ο επόμενος ερευνητής βλέπει ακριβώς τι μετρήθηκε.

> 🧠 **Σκέψη για τον φοιτητή:** Σε κάθε δικό σας project, σχεδιάζετε από την αρχή «τι αναφορά θα παράγω;». Αυτό συχνά διαμορφώνει σωστά και τον υπόλοιπο κώδικα.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          GENERATING COMPLETE STATISTICAL REPORT                ║
╚════════════════════════════════════════════════════════════════╝
""")

def generate_complete_dicom_report(dicom_obj, save_path=None):
    """
    Generate a complete statistical and quality report for a DICOM image
    
    Parameters:
    -----------
    dicom_obj : pydicom.dataset.FileDataset
        DICOM object to analyze
    save_path : str, optional
        Path to save the report
    
    Returns:
    --------
    dict : Complete report data
    """
    
    img = dicom_obj.pixel_array
    
    # Collect all analyses
    report = {
        'metadata': {},
        'basic_stats': {},
        'histogram_stats': {},
        'quality_metrics': {},
        'spatial_stats': {}
    }
    
    # Metadata
    report['metadata'] = {
        'Modality': dicom_obj.get('Modality', 'N/A'),
        'Patient_ID': dicom_obj.get('PatientID', 'N/A'),
        'Study_Date': dicom_obj.get('StudyDate', 'N/A'),
        'Image_Size': f"{dicom_obj.Rows} x {dicom_obj.Columns}",
        'Pixel_Spacing': str(dicom_obj.get('PixelSpacing', 'N/A')),
        'Slice_Thickness': str(dicom_obj.get('SliceThickness', 'N/A'))
    }
    
    # Basic statistics
    flat_img = img.flatten()
    report['basic_stats'] = {
        'Mean': float(flat_img.mean()),
        'Median': float(np.median(flat_img)),
        'Std_Dev': float(flat_img.std()),
        'Variance': float(flat_img.var()),
        'Min': float(flat_img.min()),
        'Max': float(flat_img.max()),
        'Range': float(flat_img.ptp()),
        'Q1': float(np.percentile(flat_img, 25)),
        'Q3': float(np.percentile(flat_img, 75)),
        'IQR': float(np.percentile(flat_img, 75) - np.percentile(flat_img, 25))
    }
    
    # Histogram statistics
    from scipy import stats as scipy_stats
    report['histogram_stats'] = {
        'Skewness': float(scipy_stats.skew(flat_img)),
        'Kurtosis': float(scipy_stats.kurtosis(flat_img)),
        'CV_Percent': float((flat_img.std() / flat_img.mean()) * 100) if flat_img.mean() != 0 else 0,
    }
    
    # Quality metrics
    noise_metrics = estimate_noise_level(dicom_obj)
    report['quality_metrics'] = {
        'SNR': float(noise_metrics['snr']),
        'SNR_dB': float(noise_metrics['snr_db']),
        'Estimated_Noise_Level': float(noise_metrics['estimated_std_from_mad']),
        'MAD': float(noise_metrics['mad'])
    }
    
    # Spatial statistics
    img_norm = (img - img.min()) / (img.max() - img.min())
    report['spatial_stats'] = {
        'Brightness': float(img_norm.mean()),
        'Contrast_RMS': float(img_norm.std()),
        'Dynamic_Range': float(img_norm.ptp())
    }
    
    # Create comprehensive visualization
    fig = plt.figure(figsize=(18, 14))
    gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.3)
    
    # 1. Main image
    ax1 = fig.add_subplot(gs[0:2, 0:2])
    ax1.imshow(img, cmap='gray')
    ax1.set_title(f'DICOM Image - {report["metadata"]["Modality"]}', 
                 fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # 2. Histogram
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.hist(flat_img, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(report['basic_stats']['Mean'], color='red', linestyle='--', 
               linewidth=2, label=f"Mean: {report['basic_stats']['Mean']:.1f}")
    ax2.axvline(report['basic_stats']['Median'], color='green', linestyle='--',
               linewidth=2, label=f"Median: {report['basic_stats']['Median']:.1f}")
    ax2.set_title('Intensity Distribution', fontsize=10, fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # 3. Box plot
    ax3 = fig.add_subplot(gs[1, 2])
    ax3.boxplot(flat_img, vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue'))
    ax3.set_title('Box Plot', fontsize=10, fontweight='bold')
    ax3.set_ylabel('Intensity')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Metadata table
    ax4 = fig.add_subplot(gs[2, 0])
    ax4.axis('off')
    metadata_text = "METADATA\n" + "─"*30 + "\n"
    for key, value in report['metadata'].items():
        metadata_text += f"{key.replace('_', ' ')}: {value}\n"
    ax4.text(0.1, 0.5, metadata_text, fontsize=9, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))
    
    # 5. Basic statistics table
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')
    stats_text = "BASIC STATISTICS\n" + "─"*30 + "\n"
    for key, value in report['basic_stats'].items():
        stats_text += f"{key.replace('_', ' ')}: {value:.2f}\n"
    ax5.text(0.1, 0.5, stats_text, fontsize=9, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))
    
    # 6. Quality metrics table
    ax6 = fig.add_subplot(gs[2, 2])
    ax6.axis('off')
    quality_text = "QUALITY METRICS\n" + "─"*30 + "\n"
    for key, value in report['quality_metrics'].items():
        quality_text += f"{key.replace('_', ' ')}: {value:.2f}\n"
    ax6.text(0.1, 0.5, quality_text, fontsize=9, family='monospace',
            verticalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))
    
    # 7. Intensity profile
    ax7 = fig.add_subplot(gs[3, :])
    mid_row = img.shape[0] // 2
    profile = img[mid_row, :]
    ax7.plot(profile, linewidth=2, color='darkblue')
    ax7.fill_between(range(len(profile)), profile, alpha=0.3)
    ax7.set_title(f'Intensity Profile (Row {mid_row})', fontsize=10, fontweight='bold')
    ax7.set_xlabel('Column Position')
    ax7.set_ylabel('Intensity')
    ax7.grid(True, alpha=0.3)
    
    # Overall title
    plt.suptitle('COMPREHENSIVE DICOM IMAGE ANALYSIS REPORT', 
                fontsize=18, fontweight='bold', y=0.98)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✓ Report saved to: {save_path}")
    
    plt.show()
    
    return report

# Generate complete report
print("\n" + "="*70)
print("GENERATING COMPREHENSIVE REPORT...")
print("="*70)

report_data = generate_complete_dicom_report(dicom_data)

# Print summary
print("\n📋 REPORT SUMMARY:")
print(json.dumps(report_data, indent=2))


## Ενότητα 35 — Ασκήσεις Πρακτικής Εξάσκησης

### Γιατί ασκήσεις;

Δεν μαθαίνεται η ανάλυση εικόνων διαβάζοντας. Μαθαίνεται **τρέχοντας κώδικα, σπάζοντάς τον, και διορθώνοντας τα σφάλματα**. Οι παρακάτω οκτώ ασκήσεις είναι σχεδιασμένες ώστε να καλύπτουν διαφορετικές πτυχές αυτού που μάθαμε:

| # | Άσκηση | Τι εξασκεί |
|---|--------|-----------|
| 1 | Στατιστική Ανάλυση | Βασική ποσοτικοποίηση + έλεγχος κανονικότητας |
| 2 | Manipulation Ιστογράμματος | Επίδραση παραμέτρων στην ερμηνεία |
| 3 | Region Analysis | Χωρική ανάλυση + heatmaps |
| 4 | Noise Assessment | Κριτική σύγκριση μεθόδων |
| 5 | Profile Analysis | Μέτρηση σε φυσικές μονάδες (mm) |
| 6 | Comparative Analysis | Σύγκριση modalities |
| 7 | Quality Control | Σχεδιασμός pipeline |
| 8 | Advanced Visualization | Σύνθεση παρουσίασης |

### Συμβουλές για να δουλέψετε σωστά

> 📌 **Πάντοτε:**
> 1. **Επικυρώνετε τα δεδομένα εισόδου** — το πρώτο πράγμα που πρέπει να ελέγχετε είναι αν το DICOM φορτώθηκε σωστά (`dicom_obj.pixel_array.shape`, `dicom_obj.Modality`).
> 2. **Σχολιάζετε τον κώδικα** — γράψτε τι κάνετε ΚΑΙ γιατί. Σε έναν μήνα δεν θα το θυμάστε.
> 3. **Συγκρίνετε με αναμενόμενες τιμές** — π.χ. αν η μέση τιμή ενός CT φόντου σας βγει 500, κάτι πάει στραβά (φόντο = αέρας ≈ −1000 HU).
> 4. **Οπτικοποιείτε ενδιάμεσα βήματα** — μην αφήνετε τίποτα ως «μαύρο κουτί».
> 5. **Σκέφτεστε την κλινική σημασία** — δεν φτιάχνουμε γραφήματα για τα γραφήματα. Πάντα ρωτάτε «τι θα έλεγε ένας γιατρός βλέποντας αυτό;».

### Πώς να παραδώσετε

Σε κάθε άσκηση ζητείται:
- **Λειτουργικός κώδικας** σε notebook ή script.
- **Σχόλια** που εξηγούν τη λογική.
- **Ποιοτική ερμηνεία** των αποτελεσμάτων (όχι μόνο αριθμοί — τι σημαίνουν).

> 💡 Η καλύτερη παράδοση είναι αυτή που, αν τη δει κάποιος που δεν ξέρει DICOM, θα την καταλάβει πλήρως.


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║              ENHANCED PRACTICE EXERCISES                       ║
╚════════════════════════════════════════════════════════════════╝

📝 EXERCISE 1: Statistical Analysis
   Task: Load a DICOM image and:
   a) Calculate all central tendency measures (mean, median, mode)
   b) Calculate all dispersion measures (std, variance, range, IQR)
   c) Determine if the distribution is normal (using Q-Q plot)
   d) Identify and explain any outliers

📝 EXERCISE 2: Histogram Manipulation
   Task: 
   a) Create histograms with different bin sizes (10, 50, 100, 200)
   b) Compare how bin size affects interpretation
   c) Apply histogram equalization
   d) Compare before/after statistics

📝 EXERCISE 3: Region Analysis
   Task:
   a) Divide image into 4x4 grid
   b) Calculate mean and std for each region
   c) Create heatmaps showing spatial variation
   d) Identify regions with highest/lowest contrast

📝 EXERCISE 4: Noise Assessment
   Task:
   a) Estimate noise level using multiple methods
   b) Calculate SNR
   c) Compare corner regions for noise uniformity
   d) Suggest if image quality is adequate

📝 EXERCISE 5: Profile Analysis
   Task:
   a) Extract horizontal, vertical, and diagonal profiles
   b) Calculate gradients (rate of change)
   c) Identify edges and transitions
   d) Measure distances in physical units (mm)

📝 EXERCISE 6: Comparative Analysis
   Task:
   a) Load 3 different DICOM images (different modalities if possible)
   b) Create comparative statistics table
   c) Generate histograms side-by-side
   d) Explain differences based on modality

📝 EXERCISE 7: Quality Control
   Task:
   a) Create a quality control pipeline
   b) Check for: artifacts, proper contrast, noise levels
   c) Generate automatic quality report
   d) Flag images that fail quality criteria

📝 EXERCISE 8: Advanced Visualization
   Task:
   a) Create a dashboard with: image, histogram, statistics, profiles
   b) Add interactive elements (if possible)
   c) Include both spatial and statistical views
   d) Export as a report

💡 TIPS FOR EXERCISES:
   - Always validate your inputs
   - Document your code with comments
   - Compare results with expected values
   - Visualize intermediate steps
   - Think about clinical relevance
""")


## Ενότητα 36 — Συνοπτικός Οδηγός Αναφοράς

### Τι είναι αυτή η ενότητα;

Ένα **cheat sheet** που μπορείτε να έχετε δίπλα σας όταν προγραμματίζετε. Μαζευτήκαμε σε ένα μέρος όλες τις βασικές συναρτήσεις που χρησιμοποιήσαμε — οργανωμένες ανά κατηγορία.

### Πώς να το χρησιμοποιείτε

- **Πρώτη γραμμή άμυνας:** Όταν δεν θυμάστε πώς υπολογίζεται κάτι, κοιτάτε εδώ πριν ψάξετε στο Google.
- **Reference για ασκήσεις και projects:** Αντιγράφετε τα μοτίβα και τα προσαρμόζετε.
- **Σύνδεση εννοιών με κώδικα:** Κάθε στατιστική έννοια (μέση τιμή, IQR, kurtosis...) έχει τη δική της κλήση συνάρτησης.

### Οι κατηγορίες

| Κατηγορία | Τι περιλαμβάνει |
|-----------|------------------|
| **Central Tendency** | mean, median, mode |
| **Dispersion** | std, variance, range, IQR, CV |
| **Percentiles** | quartiles και custom percentiles |
| **Distribution Shape** | skewness, kurtosis |
| **Histogram** | basic, normalized, cumulative |
| **Image Quality** | SNR (linear & dB), contrast, brightness |
| **DICOM Access** | direct, safe (`.get`), tag access, pixel access |
| **Common Operations** | normalize, window, threshold, gradient |

### Επόμενα βήματα μετά αυτό το μάθημα

Έχετε πλέον τις βάσεις για να εξερευνήσετε:

1. **Segmentation** — αυτόματη οριοθέτηση οργάνων ή βλαβών.
2. **Registration** — ευθυγράμμιση εικόνων διαφορετικών εξετάσεων.
3. **Radiomics** — εξαγωγή εκατοντάδων χαρακτηριστικών για ML.
4. **Deep Learning σε ιατρικές εικόνες** — CNN, U-Net, transformers.
5. **Multi-modal ανάλυση** — συνδυασμός CT/MRI/PET.

> 🎓 **Τελική σκέψη:** Η ανάλυση ιατρικών εικόνων είναι μια από τις περιοχές της AI με μεγαλύτερο **κλινικό και ηθικό βάρος**. Αυτό που σήμερα μαθαίνετε ως «παιχνίδι με αριθμούς» μπορεί αύριο να επηρεάζει διαγνώσεις. Πάντα να σκέφτεστε: σωστά δεδομένα, σωστές παραδοχές, διαφανής αναφορά.

🏥 **Καλή ανάλυση!**


In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║                  QUICK REFERENCE GUIDE                         ║
╚════════════════════════════════════════════════════════════════╝

📚 STATISTICAL MEASURES QUICK REFERENCE:

CENTRAL TENDENCY:
├─ Mean:     np.mean(array)
├─ Median:   np.median(array)
└─ Mode:     scipy.stats.mode(array)

DISPERSION:
├─ Std Dev:  np.std(array)
├─ Variance: np.var(array)
├─ Range:    np.ptp(array)  # peak-to-peak
├─ IQR:      Q3 - Q1
└─ CV:       (std/mean) * 100

PERCENTILES:
├─ Quartiles: np.percentile(array, [25, 50, 75])
├─ Custom:    np.percentile(array, p)
└─ Quantile:  np.quantile(array, q)

DISTRIBUTION SHAPE:
├─ Skewness: scipy.stats.skew(array)
└─ Kurtosis: scipy.stats.kurtosis(array)

HISTOGRAM:
├─ Basic:    plt.hist(array, bins=50)
├─ Normed:   plt.hist(array, density=True)
└─ Cumul:    np.cumsum(counts)

IMAGE QUALITY:
├─ SNR:      mean / std
├─ SNR (dB): 20 * log10(SNR)
├─ Contrast: std of normalized image
└─ Bright:   mean of normalized image

DICOM ACCESS:
├─ Direct:   dicom_obj.TagName
├─ Safe:     dicom_obj.get('TagName', default)
├─ Tag num:  dicom_obj[0x0008, 0x0060].value
└─ Pixel:    dicom_obj.pixel_array

COMMON OPERATIONS:
├─ Normalize:  (img - min) / (max - min)
├─ Window:     np.clip(img, wc-ww/2, wc+ww/2)
├─ Threshold:  img > threshold
└─ Gradient:   np.gradient(img)
""")

print("""
╔════════════════════════════════════════════════════════════════╗
║                    TUTORIAL COMPLETE!                          ║
╚════════════════════════════════════════════════════════════════╝

🎉 Congratulations! You now have comprehensive knowledge of:

✓ DICOM image fundamentals
✓ Statistical analysis techniques
✓ Histogram interpretation and manipulation
✓ Quality assessment methods
✓ Noise analysis
✓ Contrast and brightness evaluation
✓ Region-based analysis
✓ Profile analysis
✓ Complete reporting

🚀 NEXT STEPS:
1. Practice with real DICOM datasets
2. Experiment with different modalities
3. Build your own analysis tools
4. Explore advanced topics (segmentation, registration, etc.)
5. Apply these techniques to research projects

📖 Keep this notebook as a reference for future work!

Happy analyzing! 🏥💻📊
""")
